
# EnerDR-Net v4.0 — Complete Experimental APTOS → Messidor-2 / IDRiD Experimental Pipeline

**Purpose.** End-to-end implementation of EnerDR-Net and the complete paper-facing experimental protocol.  
This version extends the original v3.1 notebook so that all results used in the manuscript are generated from saved
predictions/checkpoints rather than entered manually.

**Architecture (unchanged):** Multi-scale retinal encoder → evidence particles → Relational Disease Energy Model (RDEM) →
safeguarded Learned Relational Energy Flow (LREF) → Disease-State Vector (DSV) → Ordered Disease-State Axis (ODSA) →
Dual-Energy Grade Decoder (DEGD), with photometric pathology-state consistency (PSIC).

**Added paper analyses:** IDRiD frozen external evaluation; MCE/NLL/reliability diagrams; per-class recall; ordinal-error
distribution; RDR/VTDR clinical screening metrics with Wilson confidence intervals and probability AUROC/AUPRC; selective
prediction; high-confidence severe-error analysis; DEGD burden-center alignment; LREF energy dynamics; RDEM structural
diagnostics; external paired bootstrap tests; hardware profiling; hyperparameter sensitivity sweeps; and a run-derived
reference-audit table.

**Evaluation policy.** APTOS is the development dataset. Messidor-2 and IDRiD are external datasets and are excluded from
optimization, checkpoint selection, early stopping, and architecture selection. Frozen APTOS-selected checkpoints are used
for all external evaluations.

> Important: all manuscript tables should be generated from the CSV files written by this notebook. The `REFERENCE_RESULTS`
> block near the end is only a regression/audit check against the supplied result specification; it is never used to create
> predictions or metrics.


In [1]:
# ============================== 1. Imports + configuration ==============================
from pathlib import Path
from dataclasses import dataclass, asdict
from contextlib import contextmanager
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import math, random, time, json, hashlib, gc, platform, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.transforms import InterpolationMode

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    cohen_kappa_score, f1_score, balanced_accuracy_score, accuracy_score,
    mean_absolute_error, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, recall_score, precision_score,
)
import matplotlib.pyplot as plt

# ------------------------------ dataset paths ------------------------------
APTOS_IMAGE_DIR = Path(r"D:\Datasets\Aptos-2019\train_images")
APTOS_CSV       = Path(r"D:\Datasets\Aptos-2019\train.csv")

MESSIDOR_IMAGE_DIR = Path(r"D:\Datasets\Messidor_02\messidor-2\images")
MESSIDOR_CSV       = Path(r"D:\Datasets\Messidor_02\messidor-2\messidor_data.csv")


IDRID_ROOT = Path(r"D:\Datasets\IDRiD\Images\Images")
IDRID_CSV = Path(r"D:\Datasets\IDRiD\idrid_labels.csv")

# Change only this output path if desired.
OUT_DIR = Path(r"D:\Projects\EnerDRNet_v4_PAPER_COMPLETE")
CACHE_DIR = OUT_DIR / "cache"
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
PREDICTION_DIR = OUT_DIR / "predictions"
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
PROFILE_DIR = OUT_DIR / "profiles"
for p in [OUT_DIR, CACHE_DIR, CHECKPOINT_DIR, PREDICTION_DIR, TABLE_DIR, FIGURE_DIR, PROFILE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------ experiment configuration ------------------------------
NUM_CLASSES = 5
IMAGE_SIZE = 320
NUM_PARTICLES = 24
PARTICLE_DIM = 64
FLOW_STEPS = 3

BATCH_SIZE = 4
ACCUM_STEPS = 4
NUM_WORKERS = 0                    # safest on Windows/Jupyter; increase after validation
EPOCHS = 60
PATIENCE = 12
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0
WARMUP_EPOCHS = 5

SPLIT_SEED = 2026
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
FINAL_SEEDS = [17, 29, 42, 73, 101]
ABLATION_SEEDS = FINAL_SEEDS
BASELINE_SEEDS = FINAL_SEEDS
SENSITIVITY_SEEDS = FINAL_SEEDS

# Mild resampling only; no second class-weight correction in focal loss.
SAMPLER_POWER = 0.50
FOCAL_GAMMA = 1.5

# Safeguarded LREF constants.
FLOW_MAX_STEP = 0.08
FLOW_KAPPA = 0.02                  # required first-order descent component
ARMIJO_C = 0.10
BACKTRACK_FACTOR = 0.50
MAX_BACKTRACKS = 5

# RDEM coefficients.
RDEM_INTERACTION_WEIGHT = 0.15
RDEM_REPULSION_WEIGHT = 0.04
RDEM_ACTIVITY_WEIGHT = 0.001

# Paper analysis / reporting.
PREPROCESS_VERSION = "RSPIN_v4_supportcrop_squarepad_png"
RUN_EXACT_HASH_AUDIT = True
BOOTSTRAP_REPEATS = 2000
CALIBRATION_BINS = 15
SELECTIVE_COVERAGES = [1.00, 0.95, 0.90, 0.80, 0.70]
SELECTIVE_UNCERTAINTY = "1-maxprob"   # "1-maxprob" or "entropy"
HIGH_CONFIDENCE_THRESHOLD = 0.90
ACTIVE_PARTICLE_THRESHOLD = 0.50

# Dataset integrity is validated from the files actually loaded.
# No target metric values or manuscript result values are embedded in this notebook.
REQUIRED_GRADES = tuple(range(NUM_CLASSES))
STRICT_COMPLETE_RUN = True


# Expensive blocks.
RUN_FULL_EXPERIMENT = True
RUN_ABLATIONS = True
RUN_BASELINES = True
RUN_INTERPRETABILITY = True
RUN_PAPER_ANALYSES = True
RUN_EFFICIENCY_PROFILE = True
RUN_SENSITIVITY = True             # 17 configurations × five seeds is very expensive; enable for final sensitivity run
EVALUATE_EXTERNAL_FOR_ABLATIONS = False
BASELINE_PRETRAINED = True
BASELINE_NAMES = ["ResNet50", "EfficientNetB3", "ConvNeXtTiny", "SwinTiny"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT_DIR)

# Serialize scalar/list configuration. Paths are recorded separately in the final manifest.
CONFIG = {
    k: v for k, v in list(globals().items())
    if k.isupper() and isinstance(v, (int, float, str, bool, list, tuple))
}
(OUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str), encoding="utf-8")


C:\Users\Testing\.conda\envs\test_env_gpu\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda
PyTorch: 2.7.1+cu118
GPU: NVIDIA GeForce RTX 4070 SUPER
Output: D:\Projects\EnerDRNet_v4_PAPER_COMPLETE


1709

In [2]:
# ============================== 2. Reproducibility utilities ==============================
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def inverse_softplus(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Stable inverse of softplus for positive x."""
    x = x.clamp_min(eps)
    return x + torch.log(-torch.expm1(-x))


seed_everything(FINAL_SEEDS[0])

In [3]:
# ============================== 3. Dataset discovery + strict audit ==============================
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def _first_column(df: pd.DataFrame, candidates, required=True):
    lower = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise KeyError(f"None of the expected columns {candidates} exist. Available columns: {list(df.columns)}")
    return None


def build_image_index(root: Path):
    if not root.exists():
        raise FileNotFoundError(root)
    index = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES:
            for key in {p.name.lower(), p.stem.lower()}:
                index.setdefault(key, []).append(p)
    return index


def resolve_image(index, image_id):
    token = str(image_id).strip()
    candidates = []
    for key in [token.lower(), Path(token).name.lower(), Path(token).stem.lower()]:
        candidates.extend(index.get(key, []))
    unique, seen = [], set()
    for p in candidates:
        s = str(p.resolve())
        if s not in seen:
            unique.append(p)
            seen.add(s)
    if len(unique) == 1:
        return unique[0]
    if len(unique) == 0:
        return None
    # Prefer an exact filename match before declaring ambiguity.
    exact = [p for p in unique if p.name.lower() == Path(token).name.lower()]
    if len(exact) == 1:
        return exact[0]
    raise RuntimeError(f"Ambiguous image id '{image_id}' resolves to multiple files: {unique[:5]}")


def _sanitize_grade(x):
    if pd.isna(x):
        return None
    try:
        g = int(float(x))
    except Exception:
        return None
    return g if 0 <= g < NUM_CLASSES else None


def _validate_dataset_frame(name, df):
    """Validate the dataset that was actually discovered on disk.

    This checks completeness and consistency without comparing against any pre-entered
    manuscript result or target count.
    """
    if len(df) == 0:
        raise RuntimeError(f"{name}: no labelled images were loaded.")
    required = {"image_id", "path", "grade"}
    missing = required.difference(df.columns)
    if missing:
        raise RuntimeError(f"{name}: missing required columns: {sorted(missing)}")
    bad_grades = sorted(set(df["grade"].astype(int)) - set(REQUIRED_GRADES))
    if bad_grades:
        raise RuntimeError(f"{name}: invalid grades found: {bad_grades}")
    present = set(df["grade"].astype(int).unique())
    missing_grades = sorted(set(REQUIRED_GRADES) - present)
    if missing_grades:
        raise RuntimeError(f"{name}: grades absent from the loaded data: {missing_grades}")
    if df["image_id"].astype(str).duplicated().any():
        dup = df.loc[df["image_id"].astype(str).duplicated(), "image_id"].astype(str).head(10).tolist()
        raise RuntimeError(f"{name}: duplicate image IDs detected, e.g. {dup}")
    missing_paths = [str(p) for p in df["path"] if not Path(p).exists()]
    if missing_paths:
        raise FileNotFoundError(f"{name}: {len(missing_paths)} image files are missing; first: {missing_paths[0]}")
    return df

def load_aptos():
    if not APTOS_CSV.exists():
        raise FileNotFoundError(APTOS_CSV)
    raw = pd.read_csv(APTOS_CSV)
    id_col = _first_column(raw, ["id_code", "image_id", "image", "filename", "file_name"])
    grade_col = _first_column(raw, ["diagnosis", "grade", "dr_grade", "adjudicated_dr_grade"])
    index = build_image_index(APTOS_IMAGE_DIR)

    rows, missing, bad_labels = [], [], []
    for _, r in raw.iterrows():
        grade = _sanitize_grade(r[grade_col])
        if grade is None:
            bad_labels.append(r[id_col])
            continue
        p = resolve_image(index, r[id_col])
        if p is None:
            missing.append(r[id_col])
            continue
        rows.append({"image_id": str(r[id_col]), "path": str(p), "grade": grade, "dataset": "APTOS"})

    if missing:
        raise RuntimeError(f"APTOS has {len(missing)} labelled records with unresolved images. Examples: {missing[:10]}")
    df = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)
    if df.empty:
        raise RuntimeError("No valid APTOS samples were loaded.")
    print(f"APTOS raw rows={len(raw)}, valid={len(df)}, invalid_labels={len(bad_labels)}")
    print(df.grade.value_counts().sort_index().to_dict())
    _validate_dataset_frame("APTOS", df)
    return df


def load_messidor():
    if not MESSIDOR_CSV.exists():
        raise FileNotFoundError(MESSIDOR_CSV)
    raw = pd.read_csv(MESSIDOR_CSV)
    id_col = _first_column(raw, [
        "image_id", "image", "image_name", "filename", "file_name", "id_code", "name"
    ])
    grade_col = _first_column(raw, [
        "adjudicated_dr_grade", "diagnosis", "grade", "dr_grade", "retinopathy_grade", "dr"
    ])
    index = build_image_index(MESSIDOR_IMAGE_DIR)

    rows, missing, bad_labels = [], [], []
    for _, r in raw.iterrows():
        grade = _sanitize_grade(r[grade_col])
        if grade is None:
            bad_labels.append(r[id_col])
            continue
        p = resolve_image(index, r[id_col])
        if p is None:
            missing.append(r[id_col])
            continue
        rows.append({"image_id": str(r[id_col]), "path": str(p), "grade": grade, "dataset": "Messidor-2"})

    if missing:
        raise RuntimeError(
            f"Messidor-2 has {len(missing)} labelled records with unresolved images. Examples: {missing[:10]}"
        )
    df = pd.DataFrame(rows).drop_duplicates(subset=["path"]).reset_index(drop=True)
    if df.empty:
        raise RuntimeError("No valid Messidor-2 samples were loaded.")
    print(f"Messidor-2 raw rows={len(raw)}, valid={len(df)}, invalid/unlabelled={len(bad_labels)}")
    print(df.grade.value_counts().sort_index().to_dict())
    _validate_dataset_frame("Messidor-2", df)
    return df


def _idrid_candidate_csvs(root: Path):
    if IDRID_CSV is not None:
        p = Path(IDRID_CSV)
        if not p.exists():
            raise FileNotFoundError(p)
        return [p]

    csvs = list(root.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV files found under {root}")

    # Prefer disease-grading/retinopathy label files but still inspect all CSVs.
    def score(p):
        s = str(p).lower()
        return (
            5 * ("grading" in s)
            + 4 * ("retinopathy" in s)
            + 3 * ("label" in s or "groundtruth" in s or "ground_truth" in s)
            + 1 * ("train" in s or "test" in s)
        )
    return sorted(csvs, key=score, reverse=True)


def load_idrid():
    if not IDRID_ROOT.exists():
        raise FileNotFoundError(IDRID_ROOT)

    image_index = build_image_index(IDRID_ROOT)
    rows, used_csvs, unresolved = [], [], []

    id_candidates = [
        "image name", "image_name", "image", "image_id", "id_code",
        "filename", "file_name", "name"
    ]
    grade_candidates = [
        "retinopathy grade", "retinopathy_grade", "diagnosis", "grade",
        "dr_grade", "adjudicated_dr_grade", "dr"
    ]

    for csv_path in _idrid_candidate_csvs(IDRID_ROOT):
        try:
            raw = pd.read_csv(csv_path)
        except Exception:
            continue
        id_col = _first_column(raw, id_candidates, required=False)
        grade_col = _first_column(raw, grade_candidates, required=False)
        if id_col is None or grade_col is None:
            continue

        valid_here = 0
        for _, r in raw.iterrows():
            grade = _sanitize_grade(r[grade_col])
            if grade is None:
                continue
            p = resolve_image(image_index, r[id_col])
            if p is None:
                unresolved.append((str(csv_path), str(r[id_col])))
                continue
            rows.append({
                "image_id": str(r[id_col]),
                "path": str(p),
                "grade": grade,
                "dataset": "IDRiD",
                "label_source": str(csv_path),
            })
            valid_here += 1
        if valid_here:
            used_csvs.append((str(csv_path), valid_here))

    if not rows:
        raise RuntimeError(
            "No usable 5-grade IDRiD labels were discovered. Set IDRID_CSV to the exact curated grading CSV."
        )

    df = pd.DataFrame(rows)
    # If train/test label CSVs overlap or duplicate the same image, retain a single image record.
    conflicts = df.groupby("path")["grade"].nunique()
    bad = conflicts[conflicts > 1]
    if len(bad):
        raise RuntimeError(f"IDRiD contains conflicting grades for {len(bad)} image paths.")
    df = df.drop_duplicates(subset=["path"], keep="first").reset_index(drop=True)

    print("IDRiD label files used:")
    for p, n in used_csvs:
        print("  ", n, p)
    print(f"IDRiD valid={len(df)}, unresolved labelled rows={len(unresolved)}")
    if unresolved:
        print("First unresolved IDRiD records:", unresolved[:10])
    print(df.grade.value_counts().sort_index().to_dict())
    _validate_dataset_frame("IDRiD", df)
    return df


APTOS = load_aptos()
MESSIDOR = load_messidor()
IDRID = load_idrid()

DATASET_SUMMARY = pd.DataFrame([
    {"dataset": "APTOS", "total": len(APTOS), **{f"G{k}": int((APTOS.grade == k).sum()) for k in range(NUM_CLASSES)}},
    {"dataset": "Messidor-2", "total": len(MESSIDOR), **{f"G{k}": int((MESSIDOR.grade == k).sum()) for k in range(NUM_CLASSES)}},
    {"dataset": "IDRiD", "total": len(IDRID), **{f"G{k}": int((IDRID.grade == k).sum()) for k in range(NUM_CLASSES)}},
])
DATASET_SUMMARY.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
print(DATASET_SUMMARY.to_string(index=False))


APTOS raw rows=3662, valid=3662, invalid_labels=0
{0: 1805, 1: 370, 2: 999, 3: 193, 4: 295}
Messidor-2 raw rows=1748, valid=1744, invalid/unlabelled=4
{0: 1017, 1: 270, 2: 347, 3: 75, 4: 35}
IDRiD label files used:
   455 D:\Datasets\IDRiD\idrid_labels.csv
IDRiD valid=455, unresolved labelled rows=0
{0: 129, 1: 22, 2: 156, 3: 84, 4: 64}
   dataset  total   G0  G1  G2  G3  G4
     APTOS   3662 1805 370 999 193 295
Messidor-2   1744 1017 270 347  75  35
     IDRiD    455  129  22 156  84  64


In [5]:
# ============================== 4. APTOS stratified split ==============================
# Fixed APTOS split.
# Training seeds change initialization, augmentation, and shuffling,
# while the train/validation/test partition remains fixed.

_temp_frac = VAL_FRAC + TEST_FRAC

# ---------------------------------------------------------
# Train vs temporary validation/test split
# ---------------------------------------------------------
APTOS_TRAIN, APTOS_TEMP = train_test_split(
    APTOS,
    test_size=_temp_frac,
    stratify=APTOS["grade"],
    random_state=SPLIT_SEED,
)

# ---------------------------------------------------------
# Validation vs test split
# ---------------------------------------------------------
APTOS_VAL, APTOS_TEST = train_test_split(
    APTOS_TEMP,
    test_size=TEST_FRAC / _temp_frac,
    stratify=APTOS_TEMP["grade"],
    random_state=SPLIT_SEED,
)

# Reset indices
APTOS_TRAIN = APTOS_TRAIN.reset_index(drop=True)
APTOS_VAL   = APTOS_VAL.reset_index(drop=True)
APTOS_TEST  = APTOS_TEST.reset_index(drop=True)

# ---------------------------------------------------------
# Display split information
# ---------------------------------------------------------
print("=" * 70)
print("APTOS DATA SPLIT")
print("=" * 70)

print(
    f"Train      : {len(APTOS_TRAIN):,}\n"
    f"Validation : {len(APTOS_VAL):,}\n"
    f"Test       : {len(APTOS_TEST):,}\n"
    f"Total      : {len(APTOS_TRAIN) + len(APTOS_VAL) + len(APTOS_TEST):,}"
)

print("\nClass distribution:")

for name, df in [
    ("Train", APTOS_TRAIN),
    ("Validation", APTOS_VAL),
    ("Test", APTOS_TEST),
]:
    counts = (
        df["grade"]
        .value_counts()
        .sort_index()
        .reindex(range(NUM_CLASSES), fill_value=0)
        .astype(int)
        .to_dict()
    )

    print(f"{name:10s}: {counts}")

# ---------------------------------------------------------
# Basic split integrity checks
# ---------------------------------------------------------
assert len(APTOS_TRAIN) > 0
assert len(APTOS_VAL) > 0
assert len(APTOS_TEST) > 0

assert (
    len(APTOS_TRAIN)
    + len(APTOS_VAL)
    + len(APTOS_TEST)
    == len(APTOS)
), "APTOS split sizes do not sum to the original dataset size."

# Verify all DR grades remain represented in every subset
expected_grades = set(range(NUM_CLASSES))

for name, df in [
    ("train", APTOS_TRAIN),
    ("validation", APTOS_VAL),
    ("test", APTOS_TEST),
]:
    present_grades = set(df["grade"].unique())

    missing_grades = expected_grades - present_grades

    if missing_grades:
        raise RuntimeError(
            f"{name} split is missing DR grade(s): {sorted(missing_grades)}"
        )

print("\nAPTOS stratified split created successfully.")
print("=" * 70)

# Compatibility variable used by later notebook cells
HASH_AUDIT_REPORT = pd.DataFrame(
    columns=["comparison", "exact_duplicate_hashes"]
)

APTOS DATA SPLIT
Train      : 2,563
Validation : 549
Test       : 550
Total      : 3,662

Class distribution:
Train     : {0: 1263, 1: 259, 2: 699, 3: 135, 4: 207}
Validation: {0: 271, 1: 55, 2: 150, 3: 29, 4: 44}
Test      : {0: 271, 1: 56, 2: 150, 3: 29, 4: 44}

APTOS stratified split created successfully.


In [6]:
# ============================== 5. Versioned deterministic RSPIN preprocessing cache ==============================
def retinal_support_preprocess(path: str, image_size=IMAGE_SIZE):
    """Retinal support crop -> square padding -> Lanczos resize.

    No label-dependent processing is used. The transform is deterministic.
    """
    with Image.open(path) as im:
        im = im.convert("RGB")
        arr = np.asarray(im)

    gray = arr.mean(axis=2)
    threshold = max(8.0, float(np.percentile(gray, 10)))
    mask = gray > threshold

    if mask.mean() >= 0.05:
        ys, xs = np.where(mask)
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        h, w = arr.shape[:2]
        my, mx = max(2, int(0.01 * h)), max(2, int(0.01 * w))
        y0, y1 = max(0, y0 - my), min(h, y1 + my)
        x0, x1 = max(0, x0 - mx), min(w, x1 + mx)
        im = Image.fromarray(arr[y0:y1, x0:x1])
    else:
        im = Image.fromarray(arr)

    w, h = im.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (0, 0, 0))
    canvas.paste(im, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((image_size, image_size), Image.Resampling.LANCZOS)


def cache_key(path: str):
    p = Path(path)
    st = p.stat()
    token = f"{p.resolve()}|{st.st_size}|{st.st_mtime_ns}|{PREPROCESS_VERSION}|{IMAGE_SIZE}"
    return hashlib.sha1(token.encode("utf-8")).hexdigest()


def cache_dataframe(df: pd.DataFrame, namespace: str):
    target_dir = CACHE_DIR / namespace
    target_dir.mkdir(parents=True, exist_ok=True)
    out = df.copy()
    cached = []
    for i, path in enumerate(out.path.astype(str)):
        key = cache_key(path)
        dst = target_dir / f"{key}.png"
        if not dst.exists():
            retinal_support_preprocess(path).save(dst, format="PNG", optimize=True)
        cached.append(str(dst))
        if (i + 1) % 500 == 0:
            print(namespace, "cached", i + 1, "/", len(out))
    out["cached_path"] = cached
    return out


APTOS_TRAIN = cache_dataframe(APTOS_TRAIN, "APTOS")
APTOS_VAL = cache_dataframe(APTOS_VAL, "APTOS")
APTOS_TEST = cache_dataframe(APTOS_TEST, "APTOS")
MESSIDOR = cache_dataframe(MESSIDOR, "MESSIDOR2")
IDRID = cache_dataframe(IDRID, "IDRID")

for name, frame in [
    ("APTOS_TRAIN", APTOS_TRAIN), ("APTOS_VAL", APTOS_VAL), ("APTOS_TEST", APTOS_TEST),
    ("MESSIDOR", MESSIDOR), ("IDRID", IDRID),
]:
    assert all(Path(p).exists() for p in frame.cached_path), f"Missing cached files in {name}"
print("Preprocessing cache ready.")


APTOS cached 500 / 2563
APTOS cached 1000 / 2563
APTOS cached 1500 / 2563
APTOS cached 2000 / 2563
APTOS cached 2500 / 2563
APTOS cached 500 / 549
APTOS cached 500 / 550
MESSIDOR2 cached 500 / 1744
MESSIDOR2 cached 1000 / 1744
MESSIDOR2 cached 1500 / 1744
Preprocessing cache ready.


In [8]:
# ============================== 6. Augmentation + datasets + loaders ==============================
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

GEOM_TF = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(
        degrees=10,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        interpolation=InterpolationMode.BILINEAR,
        fill=0,
    ),
])
PHOTO_TF = transforms.Compose([
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.08, hue=0.02),
])
TO_TENSOR_NORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
EVAL_TF = TO_TENSOR_NORM


class RetinaDataset(Dataset):
    """For PSIC, both training views share geometry and differ photometrically."""
    def __init__(self, frame: pd.DataFrame, train=False, two_views=True):
        self.frame = frame.reset_index(drop=True)
        self.train = train
        self.two_views = two_views

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row.cached_path) as im:
            im = im.convert("RGB")
            if self.train:
                g = GEOM_TF(im)
                x1 = TO_TENSOR_NORM(PHOTO_TF(g))
                if self.two_views:
                    x2 = TO_TENSOR_NORM(PHOTO_TF(g))
                    return x1, x2, int(row.grade), str(row.image_id)
                return x1, int(row.grade), str(row.image_id)
            x = EVAL_TF(im)
            return x, int(row.grade), str(row.image_id)


def make_sampler(frame, seed):
    counts = frame.grade.value_counts().to_dict()
    weights = [(1.0 / counts[int(y)]) ** SAMPLER_POWER for y in frame.grade]
    gen = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=gen)


def _eval_loader(frame):
    return DataLoader(
        RetinaDataset(frame, train=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    )


def make_enerdr_loaders(seed, include_external=False):
    train_loader = DataLoader(
        RetinaDataset(APTOS_TRAIN, train=True, two_views=True),
        batch_size=BATCH_SIZE,
        sampler=make_sampler(APTOS_TRAIN, seed),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    )
    val_loader = _eval_loader(APTOS_VAL)
    test_loader = _eval_loader(APTOS_TEST)
    # Kept for backward compatibility: this is Messidor-2.
    external_loader = _eval_loader(MESSIDOR) if include_external else None
    return train_loader, val_loader, test_loader, external_loader


def make_baseline_loaders(seed, include_external=False):
    train_loader = DataLoader(
        RetinaDataset(APTOS_TRAIN, train=True, two_views=False),
        batch_size=BATCH_SIZE,
        sampler=make_sampler(APTOS_TRAIN, seed),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    )
    val_loader = _eval_loader(APTOS_VAL)
    test_loader = _eval_loader(APTOS_TEST)
    external_loader = _eval_loader(MESSIDOR) if include_external else None
    return train_loader, val_loader, test_loader, external_loader


EXTERNAL_FRAMES = {
    "Messidor2_external": MESSIDOR,
    "IDRiD_external": IDRID,
}


## Mathematical core

### RDEM interaction symmetry

For particles $i,j$, the interaction network receives only symmetric pair features

$$z_i+z_j,\quad |z_i-z_j|,\quad \|c_i-c_j\|,\quad s_i+s_j.$$

These are unchanged when $i$ and $j$ are exchanged. Therefore the network output and the final interaction satisfy $\psi_{ij}=\psi_{ji}$. The diagonal is explicitly masked, hence $\psi_{ii}=0$.

### Safeguarded LREF

Let $g_t=\nabla_S E(S_t)$ and let $d_t$ be the learned raw flow. The code projects $d_t$ onto the half-space

$$g_t^\top \tilde d_t\le -\kappa\|g_t\|^2,$$

so $\tilde d_t$ is a first-order descent direction whenever $g_t\neq0$. A finite candidate step is accepted only if it passes an Armijo test. If every candidate is rejected, the state is unchanged. Hence **every executed refinement step satisfies**

$$E(S_{t+1})\le E(S_t).$$

The gradient used for the projection is computed on a detached probe state to avoid expensive second-order differentiation; the actual learned flow and accepted candidate retain ordinary end-to-end classification gradients.

In [9]:
# ============================== 7. Encoder + particle emitter ==============================
class ZeroMeanConv2d(nn.Conv2d):
    def forward(self, x):
        w = self.weight - self.weight.mean(dim=(-1, -2), keepdim=True)
        return F.conv2d(x, w, self.bias, self.stride, self.padding, self.dilation, self.groups)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = ZeroMeanConv2d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(min(8, cout), cout)
        self.conv2 = ZeroMeanConv2d(cout, cout, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(min(8, cout), cout)
        self.skip = nn.Identity() if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv2d(cin, cout, 1, stride=stride, bias=False), nn.GroupNorm(min(8, cout), cout)
        )

    def forward(self, x):
        y = F.gelu(self.gn1(self.conv1(x)))
        y = self.gn2(self.conv2(y))
        return F.gelu(y + self.skip(x))


class RetinaEncoder(nn.Module):
    def __init__(self, out_dim=PARTICLE_DIM):
        super().__init__()
        self.stem3 = ZeroMeanConv2d(3, 12, 3, padding=1)
        self.stem5 = ZeroMeanConv2d(3, 12, 5, padding=2)
        self.stem7 = ZeroMeanConv2d(3, 8, 7, padding=3)
        self.net = nn.Sequential(
            ResBlock(32, 48, stride=2),
            ResBlock(48, 64, stride=2),
            ResBlock(64, 96, stride=2),
            ResBlock(96, 128, stride=1),
            nn.Conv2d(128, out_dim, 1, bias=False),
            nn.GroupNorm(min(8, out_dim), out_dim),
            nn.GELU(),
        )

    def forward(self, x):
        x = torch.cat([self.stem3(x), self.stem5(x), self.stem7(x)], dim=1)
        return self.net(x)


def unpack_state(state):
    center = state[..., :2]
    spread = F.softplus(state[..., 2]) + 1e-4
    activity = F.softplus(state[..., 3])
    z = state[..., 4:]
    return center, spread, activity, z


class ParticleEmitter(nn.Module):
    """Emits unconstrained raw spread/activity state variables exactly once.

    This fixes the v2.3 double-softplus bug.
    """
    def __init__(self, n_particles=NUM_PARTICLES, dim=PARTICLE_DIM):
        super().__init__()
        self.n = n_particles
        self.attention = nn.Conv2d(dim, n_particles, 1)
        self.activity = nn.Conv2d(dim, n_particles, 1)
        self.feature_gate = nn.Conv2d(dim, dim, 1)

    def forward(self, features):
        B, D, H, W = features.shape
        pi = torch.softmax(self.attention(features).flatten(2), dim=-1).view(B, self.n, H, W)

        ys = torch.linspace(-1, 1, H, device=features.device, dtype=features.dtype)
        xs = torch.linspace(-1, 1, W, device=features.device, dtype=features.dtype)
        yy, xx = torch.meshgrid(ys, xs, indexing="ij")

        cx = (pi * xx).sum((-1, -2))
        cy = (pi * yy).sum((-1, -2))
        dx = xx[None, None] - cx[:, :, None, None]
        dy = yy[None, None] - cy[:, :, None, None]
        spread_pos = torch.sqrt((pi * (dx.square() + dy.square())).sum((-1, -2)) + 1e-6)
        spread_raw = inverse_softplus(spread_pos.clamp_min(1e-4))

        # Store RAW activity; positivity is applied once in unpack_state().
        activity_raw = (pi * self.activity(features)).sum((-1, -2))

        gated = features * torch.sigmoid(self.feature_gate(features))
        z = torch.einsum("bnhw,bdhw->bnd", pi, gated)

        state = torch.cat([
            cx[..., None], cy[..., None], spread_raw[..., None], activity_raw[..., None], z
        ], dim=-1)

        # Fixed per-forward reference for RDEM; state itself remains fully differentiable.
        reference_state = state.detach()
        return state, reference_state, pi

In [10]:
# ============================== 8. RDEM + safeguarded LREF ==============================
def offdiag_mask(n, device):
    return ~torch.eye(n, device=device, dtype=torch.bool)


def offdiag_mean(x):
    # x: [B,N,N]
    n = x.size(1)
    mask = offdiag_mask(n, x.device)[None].expand(x.size(0), -1, -1)
    return x.masked_select(mask).view(x.size(0), -1).mean(dim=1)


class SymmetricParticleInteraction(nn.Module):
    def __init__(self, dim=PARTICLE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * dim + 2, 96),
            nn.GELU(),
            nn.Linear(96, 64),
            nn.GELU(),
            nn.Linear(64, 3),
        )

    def forward(self, center, spread, activity, z):
        zi = z[:, :, None, :]
        zj = z[:, None, :, :]
        ci = center[:, :, None, :]
        cj = center[:, None, :, :]
        si = spread[:, :, None]
        sj = spread[:, None, :]

        dist = torch.sqrt((ci - cj).square().sum(-1) + 1e-6)
        pair = torch.cat([
            zi + zj,
            (zi - zj).abs(),
            dist[..., None],
            (si + sj)[..., None],
        ], dim=-1)
        raw = self.net(pair)
        alpha = torch.tanh(raw[..., 0])
        beta = torch.tanh(raw[..., 1])
        sigma = F.softplus(raw[..., 2]) + 0.08

        ai = activity[:, :, None]
        aj = activity[:, None, :]
        psi = ai * aj * (
            alpha * torch.exp(-dist.square() / (2.0 * sigma.square() + 1e-6))
            + beta / (1.0 + dist)
        )
        eye = torch.eye(center.size(1), device=center.device, dtype=torch.bool)[None]
        return psi.masked_fill(eye, 0.0)


class RDEM(nn.Module):
    def __init__(self, dim=PARTICLE_DIM):
        super().__init__()
        self.interaction = SymmetricParticleInteraction(dim)

    def forward(self, state, reference_state, use_interaction=True):
        c, s, a, z = unpack_state(state)
        c0, s0, a0, z0 = unpack_state(reference_state)

        # Reference-preserving unary energy.
        e_ref = (
            (z - z0).square().mean((-1, -2))
            + 0.10 * (c - c0).square().mean((-1, -2))
            + 0.04 * (s - s0).square().mean(-1)
            + 0.01 * (a - a0).square().mean(-1)
        )

        if use_interaction:
            psi = self.interaction(c, s, a, z)
            e_inter = -offdiag_mean(psi)
        else:
            psi = state.new_zeros(state.size(0), state.size(1), state.size(1))
            e_inter = state.new_zeros(state.size(0))

        d2 = (c[:, :, None, :] - c[:, None, :, :]).square().sum(-1)
        rep = torch.exp(-d2 / (2.0 * 0.18**2 + 1e-6))
        eye = torch.eye(c.size(1), device=c.device, dtype=torch.bool)[None]
        rep = rep.masked_fill(eye, 0.0)
        rep = offdiag_mean(rep)

        energy = (
            e_ref
            + RDEM_INTERACTION_WEIGHT * e_inter
            + RDEM_REPULSION_WEIGHT * rep
            + RDEM_ACTIVITY_WEIGHT * a.mean(-1)
        )
        return energy, psi


class LearnedRelationalFlow(nn.Module):
    def __init__(self, dim=PARTICLE_DIM):
        super().__init__()
        state_dim = dim + 4
        self.force = nn.Sequential(
            nn.Linear(state_dim + dim + 2, 160),
            nn.GELU(),
            nn.Linear(160, 160),
            nn.GELU(),
            nn.Linear(160, state_dim),
        )
        self.gate = nn.Sequential(
            nn.Linear(state_dim, 48),
            nn.GELU(),
            nn.Linear(48, 1),
            nn.Sigmoid(),
        )

    def forward(self, state, reference_state, psi):
        ref_z = reference_state[..., 4:]
        rel_mean = psi.mean(-1, keepdim=True)
        rel_mag = psi.abs().mean(-1, keepdim=True)
        feat = torch.cat([state, ref_z, rel_mean, rel_mag], dim=-1)
        return self.gate(state) * torch.tanh(self.force(feat))


def project_particle_state(state):
    center = state[..., :2].clamp(-1.25, 1.25)
    # raw spread/activity/latent values are bounded for numerical stability.
    rest = state[..., 2:].clamp(-8.0, 8.0)
    return torch.cat([center, rest], dim=-1)


class SafeguardedRefinement(nn.Module):
    """Learned flow + descent-half-space projection + Armijo safeguard."""
    def __init__(self, energy_model: RDEM, dim=PARTICLE_DIM, steps=FLOW_STEPS):
        super().__init__()
        self.energy_model = energy_model
        self.flow = LearnedRelationalFlow(dim)
        self.eta_raw = nn.Parameter(torch.full((steps,), -2.0))
        self.steps = steps

    def _energy_gradient_probe(self, state, reference_state, use_interaction):
        # Detached probe avoids second-order backpropagation while still computing dE/dS.
        with torch.enable_grad():
            probe = state.detach().requires_grad_(True)
            e_probe, _ = self.energy_model(probe, reference_state.detach(), use_interaction)
            grad = torch.autograd.grad(e_probe.sum(), probe, create_graph=False, retain_graph=False)[0]
        return grad.detach()

    @torch.no_grad()
    def _energy_value(self, state, reference_state, use_interaction):
        return self.energy_model(state, reference_state, use_interaction)[0]

    def forward(self, state, reference_state, use_interaction=True, enabled=True):
        if not enabled or self.steps == 0:
            energy, psi = self.energy_model(state, reference_state, use_interaction)
            return state, psi, [energy.detach()], [], []

        energy_trace = []
        direction_violations = []
        acceptance_rates = []

        for t in range(self.steps):
            # psi remains on the ordinary graph and can train RDEM through downstream losses.
            _, psi = self.energy_model(state, reference_state, use_interaction)
            e0 = self._energy_value(state.detach(), reference_state, use_interaction)
            energy_trace.append(e0.detach())

            g = self._energy_gradient_probe(state, reference_state, use_interaction)
            d_raw = self.flow(state, reference_state, psi)

            gf = g.flatten(1)
            df = d_raw.flatten(1)
            gdotd = (gf * df).sum(1)
            g2 = gf.square().sum(1)

            # Projection: g^T d <= -kappa ||g||^2.
            violation_raw = gdotd + FLOW_KAPPA * g2
            correction = F.relu(violation_raw) / (g2 + 1e-8)
            direction = d_raw - correction[:, None, None] * g
            direction_violations.append(F.relu(violation_raw) / state[0].numel())

            alpha0 = FLOW_MAX_STEP * torch.sigmoid(self.eta_raw[t])
            alpha = alpha0.expand(state.size(0))
            new_state = state
            accepted = torch.zeros(state.size(0), device=state.device, dtype=torch.bool)
            g2_mean = g2 / state[0].numel()

            for _ in range(MAX_BACKTRACKS):
                candidate = project_particle_state(state + alpha[:, None, None] * direction)
                ec = self._energy_value(candidate.detach(), reference_state, use_interaction)
                rhs = e0 - ARMIJO_C * alpha * FLOW_KAPPA * g2_mean
                ok = (~accepted) & (ec <= rhs + 1e-8)
                new_state = torch.where(ok[:, None, None], candidate, new_state)
                accepted = accepted | ok
                alpha = torch.where(accepted, alpha, alpha * BACKTRACK_FACTOR)

            # If no candidate was accepted, new_state remains exactly state: E cannot increase.
            state = new_state
            e1 = self._energy_value(state.detach(), reference_state, use_interaction)
            # Defensive numerical assertion; tolerance only handles floating-point roundoff.
            if torch.any(e1 > e0 + 2e-6):
                raise RuntimeError("Safeguarded LREF violated non-increasing energy; inspect numerical stability.")
            acceptance_rates.append(accepted.float().mean().detach())

        final_energy, final_psi = self.energy_model(state, reference_state, use_interaction)
        energy_trace.append(final_energy.detach())
        return state, final_psi, energy_trace, direction_violations, acceptance_rates

### DSV, ODSA and DEGD guarantees

The DSV components are non-negative. With $w_k=\operatorname{softplus}(\omega_k)+\epsilon>0$,

$$b=\sum_{k=1}^4 w_k s_k,$$

so $\partial b/\partial s_k=w_k>0$: the burden is strictly monotone in every disease-state component.

For ODSA, let $v$ be one learned unit vector and $\delta_k=\operatorname{softplus}(r_k)+\epsilon>0$. Then

$$A_k=A_0+\left(\sum_{j=1}^{k}\delta_j\right)v.$$

All anchors are therefore collinear, and their coordinates on $v$ are strictly increasing.

DEGD uses the same positive-gap construction for ordinal burden centers $\mu_k$, so $\mu_0<\cdots<\mu_4$ exactly.

In [11]:
# ============================== 9. DSV + ODSA + DEGD + complete model ==============================
class DiseaseStateVector(nn.Module):
    def __init__(self, dim=PARTICLE_DIM):
        super().__init__()
        self.local_head = nn.Linear(dim, 1)
        self.critical_head = nn.Sequential(nn.Linear(dim, 32), nn.GELU(), nn.Linear(32, 1))
        self.burden_weight_raw = nn.Parameter(torch.zeros(4))

    def forward(self, state, psi):
        center, spread, activity, z = unpack_state(state)
        local_evidence = F.softplus(self.local_head(z).squeeze(-1))
        particle_burden = activity * local_evidence
        local = particle_burden.mean(-1)

        # Exactly zero when psi == 0; diagonal remains excluded.
        inter = offdiag_mean(psi.abs())

        denom = particle_burden.sum(-1, keepdim=True) + 1e-6
        centroid = (particle_burden[..., None] * center).sum(1) / denom
        radial = torch.sqrt((center - centroid[:, None]).square().sum(-1) + 1e-6)
        spatial_spread = (particle_burden * radial).sum(-1) / (particle_burden.sum(-1) + 1e-6)

        critical_scores = activity * F.softplus(self.critical_head(z).squeeze(-1))
        critical = torch.logsumexp(critical_scores, dim=1) - math.log(max(z.size(1), 2))

        state_vec = torch.stack([local, inter, spatial_spread, critical], dim=1)
        burden_weights = F.softplus(self.burden_weight_raw) + 1e-3
        burden = (state_vec * burden_weights[None]).sum(1)
        return state_vec, burden, particle_burden, burden_weights


class OrderedDiseaseStateAxis(nn.Module):
    def __init__(self, state_dim=4, n_classes=NUM_CLASSES):
        super().__init__()
        self.base = nn.Parameter(torch.zeros(state_dim))
        self.axis_raw = nn.Parameter(torch.randn(state_dim) * 0.05)
        self.gap_raw = nn.Parameter(torch.zeros(n_classes - 1))
        self.scale_raw = nn.Parameter(torch.zeros(state_dim))
        self.n_classes = n_classes

    def geometry(self):
        axis = F.normalize(self.axis_raw, dim=0)
        gaps = F.softplus(self.gap_raw) + 0.05
        positions = torch.cat([gaps.new_zeros(1), torch.cumsum(gaps, dim=0)])
        anchors = self.base[None] + positions[:, None] * axis[None]
        scale = F.softplus(self.scale_raw) + 0.10
        return anchors, axis, positions, scale

    def forward(self, state_vec):
        anchors, axis, positions, scale = self.geometry()
        diff = (state_vec[:, None, :] - anchors[None, :, :]) / scale[None, None, :]
        distances = torch.sqrt(diff.square().sum(-1) + 1e-8)
        coordinate = ((state_vec - self.base[None]) * axis[None]).sum(-1)
        projection = self.base[None] + coordinate[:, None] * axis[None]
        orthogonal_residual = state_vec - projection
        return {
            "anchors": anchors,
            "axis": axis,
            "positions": positions,
            "distances": distances,
            "coordinate": coordinate,
            "orthogonal_residual": orthogonal_residual,
            "scale": scale,
        }


class DualEnergyGradeDecoder(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES):
        super().__init__()
        self.center0_raw = nn.Parameter(inverse_softplus(torch.tensor(0.10)))
        self.gap_raw = nn.Parameter(torch.zeros(n_classes - 1))
        self.sigma_raw = nn.Parameter(torch.zeros(n_classes))
        self.anchor_temp_raw = nn.Parameter(torch.tensor(0.0))
        self.mix_raw = nn.Parameter(torch.tensor(0.0))
        self.n_classes = n_classes

    def centers(self):
        base = F.softplus(self.center0_raw)
        gaps = F.softplus(self.gap_raw) + 0.05
        return torch.cat([base[None], base + torch.cumsum(gaps, dim=0)])

    def forward(self, burden, anchor_distances, use_ordinal=True, use_odsa=True):
        if not (use_ordinal or use_odsa):
            raise ValueError("At least one DEGD branch must be enabled.")

        centers = self.centers()
        sigma = F.softplus(self.sigma_raw) + 0.10
        ordinal_energy = (burden[:, None] - centers[None, :]).square() / (2 * sigma[None, :].square()) + torch.log(sigma[None, :])

        anchor_temp = F.softplus(self.anchor_temp_raw) + 0.10
        anchor_energy = anchor_distances.square() / anchor_temp

        mix = torch.sigmoid(self.mix_raw)
        if use_ordinal and use_odsa:
            total_energy = (1.0 - mix) * ordinal_energy + mix * anchor_energy
        elif use_ordinal:
            total_energy = ordinal_energy
        else:
            total_energy = anchor_energy

        logits = -total_energy
        return {
            "logits": logits,
            "probs": torch.softmax(logits, dim=1),
            "ordinal_energy": ordinal_energy,
            "anchor_energy": anchor_energy,
            "centers": centers,
            "sigmas": sigma,
            "mix": mix,
        }


@dataclass(frozen=True)
class ModelSwitches:
    use_interaction: bool = True
    use_lref: bool = True
    use_odsa: bool = True
    use_ordinal_energy: bool = True
    use_psic: bool = True


FULL_SWITCHES = ModelSwitches()
ABLATION_SWITCHES = {
    "No_LREF": ModelSwitches(use_lref=False),
    "No_Interaction": ModelSwitches(use_interaction=False),
    "No_ODSA": ModelSwitches(use_odsa=False),
    "No_Ordinal_Energy": ModelSwitches(use_ordinal_energy=False),
    "No_PSIC": ModelSwitches(use_psic=False),
}


class EnerDRNetV3(nn.Module):
    def __init__(self, switches=FULL_SWITCHES):
        super().__init__()
        self.switches = switches
        self.encoder = RetinaEncoder(PARTICLE_DIM)
        self.emitter = ParticleEmitter(NUM_PARTICLES, PARTICLE_DIM)
        self.rdem = RDEM(PARTICLE_DIM)
        self.refiner = SafeguardedRefinement(self.rdem, PARTICLE_DIM, FLOW_STEPS)
        self.dsv = DiseaseStateVector(PARTICLE_DIM)
        self.odsa = OrderedDiseaseStateAxis(4, NUM_CLASSES)
        self.decoder = DualEnergyGradeDecoder(NUM_CLASSES)

    def forward(self, x):
        feature_map = self.encoder(x)
        state0, reference_state, particle_maps = self.emitter(feature_map)

        refined_state, psi, energy_trace, direction_violations, acceptance_rates = self.refiner(
            state0,
            reference_state,
            use_interaction=self.switches.use_interaction,
            enabled=self.switches.use_lref,
        )

        state_vec, burden, particle_burden, burden_weights = self.dsv(refined_state, psi)
        axis_out = self.odsa(state_vec)
        dec = self.decoder(
            burden,
            axis_out["distances"],
            use_ordinal=self.switches.use_ordinal_energy,
            use_odsa=self.switches.use_odsa,
        )

        _, spread, activity, z = unpack_state(refined_state)
        out = {
            **dec,
            **axis_out,
            "state_vec": state_vec,
            "burden": burden,
            "particle_burden": particle_burden,
            "burden_weights": burden_weights,
            "particle_maps": particle_maps,
            "psi": psi,
            "spread": spread,
            "activity": activity,
            "z": z,
            "energy_trace": energy_trace,
            "direction_violations": direction_violations,
            "acceptance_rates": acceptance_rates,
            "feature_map": feature_map,
            "state0": state0,
            "refined_state": refined_state,
        }
        return out

In [12]:
# ============================== 10. Losses with ablation-safe normalization ==============================
def focal_loss(logits, y, gamma=FOCAL_GAMMA):
    logp = F.log_softmax(logits, dim=1)
    p = logp.exp()
    idx = torch.arange(y.size(0), device=y.device)
    pt = p[idx, y].clamp_min(1e-8)
    return (-(1.0 - pt).pow(gamma) * logp[idx, y]).mean()


def cumulative_ordinal_loss(probs, y):
    cum = torch.stack([probs[:, k:].sum(1) for k in range(1, NUM_CLASSES)], dim=1).clamp(1e-6, 1 - 1e-6)
    logits = torch.logit(cum)
    targets = torch.stack([(y >= k).float() for k in range(1, NUM_CLASSES)], dim=1)
    return F.binary_cross_entropy_with_logits(logits, targets)


def pairwise_rank_loss(burden, y, margin_per_grade=0.10):
    dy = y[:, None] - y[None, :]
    mask = dy > 0
    if not mask.any():
        return burden.new_tensor(0.0)
    required = margin_per_grade * dy.float()
    observed = burden[:, None] - burden[None, :]
    return F.relu(required - observed)[mask].mean()


def burden_center_loss(out, y):
    target = out["centers"][y]
    sigma = out["sigmas"][y].detach().clamp_min(0.1)
    return F.smooth_l1_loss((out["burden"] - target) / sigma, torch.zeros_like(target))


def anchor_margin_loss(distances, y, margin=0.25):
    target = distances.gather(1, y[:, None]).squeeze(1)
    violation = target[:, None] - distances + margin
    mask = torch.ones_like(violation, dtype=torch.bool)
    mask.scatter_(1, y[:, None], False)
    return F.relu(violation)[mask].mean()


def axis_residual_loss(out):
    # Mildly encourages disease-state vectors to be represented by the learned axis.
    return out["orthogonal_residual"].square().mean()


def symmetric_psic_loss(a, b):
    def sym(x, y):
        return 0.5 * (F.smooth_l1_loss(x, y.detach()) + F.smooth_l1_loss(y, x.detach()))
    return (
        sym(a["state_vec"], b["state_vec"])
        + 0.50 * sym(a["burden"], b["burden"])
        + 0.25 * sym(a["probs"], b["probs"])
        + 0.10 * sym(a["particle_burden"], b["particle_burden"])
    )


def particle_diversity_loss(z):
    z = F.normalize(z, dim=-1)
    sim = torch.einsum("bnd,bmd->bnm", z, z).abs()
    n = z.size(1)
    eye = torch.eye(n, device=z.device, dtype=torch.bool)[None]
    return sim.masked_select(~eye).mean()


def attention_overlap_loss(pi):
    p = F.normalize(pi.flatten(2), p=2, dim=-1)
    sim = torch.einsum("bnh,bmh->bnm", p, p)
    n = p.size(1)
    eye = torch.eye(n, device=p.device, dtype=torch.bool)[None]
    return sim.masked_select(~eye).mean()


# Weights sum to 1.0 in the Full model. For ablations, active weights are renormalized
# so deleting a module does not simply reduce the total gradient scale.
LOSS_WEIGHTS = {
    "main_cls": 0.34,
    "ordinal_aux": 0.07,
    "anchor_aux": 0.07,
    "cumulative_ordinal": 0.09,
    "burden_center": 0.08,
    "burden_rank": 0.06,
    "anchor_margin": 0.05,
    "axis_residual": 0.03,
    "flow_descent": 0.04,
    "psic": 0.06,
    "particle_diversity": 0.04,
    "attention_overlap": 0.03,
    "activity_reg": 0.04,
}
assert abs(sum(LOSS_WEIGHTS.values()) - 1.0) < 1e-9


def _average_if_two(fn, out1, out2):
    return fn(out1) if out2 is None else 0.5 * (fn(out1) + fn(out2))


def compute_loss(out1, y, switches: ModelSwitches, out2=None):
    components = {}
    active = {}

    components["main_cls"] = _average_if_two(lambda o: focal_loss(o["logits"], y), out1, out2)
    active["main_cls"] = True

    if switches.use_ordinal_energy:
        components["ordinal_aux"] = _average_if_two(lambda o: focal_loss(-o["ordinal_energy"], y), out1, out2)
        components["burden_center"] = _average_if_two(lambda o: burden_center_loss(o, y), out1, out2)
    else:
        components["ordinal_aux"] = out1["burden"].new_tensor(0.0)
        components["burden_center"] = out1["burden"].new_tensor(0.0)
    active["ordinal_aux"] = switches.use_ordinal_energy
    active["burden_center"] = switches.use_ordinal_energy

    if switches.use_odsa:
        components["anchor_aux"] = _average_if_two(lambda o: focal_loss(-o["anchor_energy"], y), out1, out2)
        components["anchor_margin"] = _average_if_two(lambda o: anchor_margin_loss(o["distances"], y), out1, out2)
        components["axis_residual"] = _average_if_two(axis_residual_loss, out1, out2)
    else:
        zero = out1["burden"].new_tensor(0.0)
        components["anchor_aux"] = zero
        components["anchor_margin"] = zero
        components["axis_residual"] = zero
    active["anchor_aux"] = switches.use_odsa
    active["anchor_margin"] = switches.use_odsa
    active["axis_residual"] = switches.use_odsa

    components["cumulative_ordinal"] = _average_if_two(lambda o: cumulative_ordinal_loss(o["probs"], y), out1, out2)
    components["burden_rank"] = _average_if_two(lambda o: pairwise_rank_loss(o["burden"], y), out1, out2)
    active["cumulative_ordinal"] = True
    active["burden_rank"] = True

    if switches.use_lref and out1["direction_violations"]:
        components["flow_descent"] = torch.stack([v.mean() for v in out1["direction_violations"]]).mean()
        if out2 is not None and out2["direction_violations"]:
            components["flow_descent"] = 0.5 * (
                components["flow_descent"] + torch.stack([v.mean() for v in out2["direction_violations"]]).mean()
            )
    else:
        components["flow_descent"] = out1["burden"].new_tensor(0.0)
    active["flow_descent"] = switches.use_lref

    if switches.use_psic and out2 is not None:
        components["psic"] = symmetric_psic_loss(out1, out2)
        active["psic"] = True
    else:
        components["psic"] = out1["burden"].new_tensor(0.0)
        active["psic"] = False

    components["particle_diversity"] = _average_if_two(lambda o: particle_diversity_loss(o["z"]), out1, out2)
    components["attention_overlap"] = _average_if_two(lambda o: attention_overlap_loss(o["particle_maps"]), out1, out2)
    components["activity_reg"] = _average_if_two(lambda o: o["activity"].mean(), out1, out2)
    active["particle_diversity"] = True
    active["attention_overlap"] = True
    active["activity_reg"] = True

    active_weight = sum(LOSS_WEIGHTS[k] for k, on in active.items() if on)
    total = sum(LOSS_WEIGHTS[k] * components[k] for k, on in active.items() if on) / active_weight
    return total, {k: float(v.detach()) for k, v in components.items()}, active_weight

In [13]:
# ============================== 11. Architectural proof/integrity checks ==============================
def run_architecture_integrity_checks():
    seed_everything(123)
    model = EnerDRNetV3(FULL_SWITCHES).to(DEVICE)
    model.train()
    x = torch.randn(2, 3, 128, 128, device=DEVICE)
    y = torch.tensor([0, 4], device=DEVICE)
    out = model(x)

    integrity = []

    # 1) Interaction symmetry and zero diagonal.
    psi = out["psi"]
    symmetry_error = float((psi - psi.transpose(1, 2)).abs().max().detach().cpu())
    diag_max = float(torch.diagonal(psi, dim1=1, dim2=2).abs().max().detach().cpu())
    symmetry_ok = symmetry_error <= 1e-5
    diag_ok = diag_max <= 1e-7
    assert symmetry_ok, "psi is not symmetric"
    assert diag_ok, "psi diagonal is not zero"
    integrity += [
        {"property": "psi_ij_equals_psi_ji", "passed": symmetry_ok, "measured": symmetry_error},
        {"property": "psi_ii_equals_0", "passed": diag_ok, "measured": diag_max},
    ]

    # 2) Zero interaction really means zero DSV interaction component.
    zero_psi = torch.zeros_like(psi)
    dummy_state = project_particle_state(
        torch.randn(2, NUM_PARTICLES, PARTICLE_DIM + 4, device=DEVICE)
    )
    sv0, _, _, _ = model.dsv(dummy_state, zero_psi)
    zero_inter_max = float(sv0[:, 1].abs().max().detach().cpu())
    zero_inter_ok = zero_inter_max <= 1e-7
    assert zero_inter_ok, "zero psi does not map to zero interaction burden"
    integrity.append({
        "property": "Psi_zero_implies_vinter_zero",
        "passed": zero_inter_ok,
        "measured": zero_inter_max,
    })

    # 3) Positive disease-burden weights => partial b / partial v_j > 0.
    burden_weights = model.dsv.burden_weight_raw
    positive_weights = F.softplus(burden_weights) + 1e-3
    min_weight = float(positive_weights.min().detach().cpu())
    weight_ok = min_weight > 0
    assert weight_ok
    integrity += [
        {"property": "positive_burden_weights", "passed": weight_ok, "measured": min_weight},
        {"property": "partial_b_partial_v_positive", "passed": weight_ok, "measured": min_weight},
    ]

    # 4) ODSA anchors are collinear and strictly ordered in axis coordinate.
    anchors, axis, pos, _ = model.odsa.geometry()
    order_ok = bool(torch.all(pos[1:] > pos[:-1]).item())
    residual = anchors - (anchors[0][None] + pos[:, None] * axis[None])
    collinear_error = float(residual.abs().max().detach().cpu())
    collinear_ok = collinear_error <= 1e-5
    assert order_ok, "ODSA positions are not strictly increasing"
    assert collinear_ok, "ODSA anchors are not collinear"
    integrity += [
        {"property": "ODSA_anchors_collinear", "passed": collinear_ok, "measured": collinear_error},
        {"property": "ODSA_grade_order_strict", "passed": order_ok, "measured": float(torch.diff(pos).min().detach().cpu())},
    ]

    # 5) DEGD centers strictly increase.
    centers = model.decoder.centers()
    center_gap = float(torch.diff(centers).min().detach().cpu())
    centers_ok = center_gap > 0
    assert centers_ok, "DEGD centers are not strictly ordered"
    integrity.append({
        "property": "mu0_lt_mu1_lt_mu2_lt_mu3_lt_mu4",
        "passed": centers_ok,
        "measured": center_gap,
    })

    # 6) Safeguarded energy trace is non-increasing sample-wise.
    trace = torch.stack(out["energy_trace"], dim=1)
    max_increase = float((trace[:, 1:] - trace[:, :-1]).max().detach().cpu()) if trace.shape[1] > 1 else 0.0
    energy_ok = max_increase <= 2e-6
    assert energy_ok, "LREF energy increased"
    integrity.append({
        "property": "E_next_le_E_current",
        "passed": energy_ok,
        "measured": max_increase,
    })

    # 7) Probabilities sum to one.
    prob_sum_error = float((out["probs"].sum(1) - 1.0).abs().max().detach().cpu())
    prob_ok = prob_sum_error <= 1e-6
    assert prob_ok
    integrity.append({
        "property": "sum_probabilities_equals_1",
        "passed": prob_ok,
        "measured": prob_sum_error,
    })

    # 8) End-to-end gradient coverage.
    loss, _, _ = compute_loss(out, y, FULL_SWITCHES, None)
    loss.backward()
    core = {
        "Encoder": model.encoder,
        "Emitter": model.emitter,
        "RDEM": model.rdem,
        "LREF_flow": model.refiner.flow,
        "DSV": model.dsv,
        "ODSA": model.odsa,
        "Decoder": model.decoder,
    }
    grad_rows = []
    for name, module in core.items():
        trainable = [p for p in module.parameters() if p.requires_grad]
        got = [p for p in trainable if p.grad is not None and torch.isfinite(p.grad).all()]
        grad_rows.append({
            "module": name,
            "parameters_with_finite_gradient": len(got),
            "trainable_parameter_tensors": len(trainable),
            "coverage": f"{len(got)}/{len(trainable)}",
        })
        if len(got) == 0:
            raise RuntimeError(f"No gradients reached core module: {name}")

    integrity_df = pd.DataFrame(integrity)
    gradient_df = pd.DataFrame(grad_rows)
    integrity_df.to_csv(TABLE_DIR / "mathematical_implementation_integrity.csv", index=False)
    gradient_df.to_csv(TABLE_DIR / "gradient_coverage.csv", index=False)

    print("Integrity checks PASSED")
    print(integrity_df.to_string(index=False))
    print("\nGradient coverage:\n", gradient_df.to_string(index=False))

    del model, x, out, loss
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return integrity_df, gradient_df


INTEGRITY_RESULTS, GRADIENT_COVERAGE = run_architecture_integrity_checks()


Integrity checks PASSED
                       property  passed      measured
           psi_ij_equals_psi_ji    True  0.000000e+00
                psi_ii_equals_0    True  0.000000e+00
   Psi_zero_implies_vinter_zero    True  0.000000e+00
        positive_burden_weights    True  6.941472e-01
   partial_b_partial_v_positive    True  6.941472e-01
         ODSA_anchors_collinear    True  0.000000e+00
        ODSA_grade_order_strict    True  7.431471e-01
mu0_lt_mu1_lt_mu2_lt_mu3_lt_mu4    True  7.431471e-01
            E_next_le_E_current    True -1.136214e-06
     sum_probabilities_equals_1    True  5.960464e-08

Gradient coverage:
    module  parameters_with_finite_gradient  trainable_parameter_tensors coverage
  Encoder                               45                           45    45/45
  Emitter                                6                            6      6/6
     RDEM                                6                            6      6/6
LREF_flow                            

In [14]:
# ============================== 12. Metrics, calibration, clinical endpoints + bootstrap ==============================
def reliability_bins(probs, y, bins=CALIBRATION_BINS):
    probs = np.asarray(probs, dtype=float)
    y = np.asarray(y, dtype=int)
    conf = probs.max(1)
    pred = probs.argmax(1)
    edges = np.linspace(0.0, 1.0, bins + 1)
    rows = []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (conf >= lo) & (conf <= hi if i == bins - 1 else conf < hi)
        count = int(mask.sum())
        if count == 0:
            continue
        acc = float((pred[mask] == y[mask]).mean())
        mean_conf = float(conf[mask].mean())
        gap = abs(acc - mean_conf)
        rows.append({"bin": i, "lo": lo, "hi": hi, "count": count, "accuracy": acc,
                     "mean_confidence": mean_conf, "abs_gap": gap})
    if not rows:
        raise RuntimeError("Calibration table has no populated bins.")
    return pd.DataFrame(rows)

def expected_calibration_error(probs, y, bins=CALIBRATION_BINS):
    tab = reliability_bins(probs, y, bins)
    n = max(int(tab["count"].sum()), 1)
    return float(np.sum((tab["count"] / n) * tab["abs_gap"]))


def maximum_calibration_error(probs, y, bins=CALIBRATION_BINS):
    tab = reliability_bins(probs, y, bins)
    vals = tab.loc[tab["count"] > 0, "abs_gap"].to_numpy(float)
    return float(np.max(vals))


def multiclass_nll(probs, y):
    probs = np.asarray(probs, dtype=float)
    y = np.asarray(y, dtype=int)
    p_true = probs[np.arange(len(y)), y].clip(1e-12, 1.0)
    return float(-np.log(p_true).mean())


def predictive_entropy(probs):
    p = np.asarray(probs, dtype=float).clip(1e-12, 1.0)
    return -(p * np.log(p)).sum(1)


def metrics_from_result(result):
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    probs = np.asarray(result["probs"], dtype=float)
    onehot = np.eye(NUM_CLASSES)[y]

    out = {
        "QWK": float(cohen_kappa_score(y, pred, weights="quadratic")),
        "Macro-F1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "Balanced-Acc": float(balanced_accuracy_score(y, pred)),
        "Accuracy": float(accuracy_score(y, pred)),
        "MAE": float(mean_absolute_error(y, pred)),
        "ECE": expected_calibration_error(probs, y),
        "MCE": maximum_calibration_error(probs, y),
        "Brier": float(np.mean(np.sum((probs - onehot) ** 2, axis=1))),
        "NLL": multiclass_nll(probs, y),
    }
    try:
        out["AUROC-OVR"] = float(roc_auc_score(y, probs, multi_class="ovr", average="macro"))
    except Exception:
        out["AUROC-OVR"] = np.nan
    try:
        out["AUPRC-OVR"] = float(average_precision_score(onehot, probs, average="macro"))
    except Exception:
        out["AUPRC-OVR"] = np.nan

    f1s = f1_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    recalls = recall_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    precisions = precision_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    for g in range(NUM_CLASSES):
        out[f"F1_grade_{g}"] = float(f1s[g])
        out[f"Recall_grade_{g}"] = float(recalls[g])
        out[f"Precision_grade_{g}"] = float(precisions[g])

    if result.get("burden") is not None:
        out["Burden-Spearman"] = float(
            pd.Series(np.asarray(result["burden"], dtype=float)).corr(pd.Series(y), method="spearman")
        )
    return out


def ordinal_error_distribution(result):
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    err = np.abs(pred - y)
    rows = []
    for d in range(NUM_CLASSES):
        rows.append({
            "error_magnitude": d,
            "count": int((err == d).sum()),
            "fraction": float((err == d).mean()),
            "percent": float(100 * (err == d).mean()),
        })
    rows.extend([
        {
            "error_magnitude": ">=2",
            "count": int((err >= 2).sum()),
            "fraction": float((err >= 2).mean()),
            "percent": float(100 * (err >= 2).mean()),
        },
        {
            "error_magnitude": "all_incorrect",
            "count": int((err >= 1).sum()),
            "fraction": float((err >= 1).mean()),
            "percent": float(100 * (err >= 1).mean()),
        },
    ])
    return pd.DataFrame(rows)


def failure_case_summary(result, high_conf=HIGH_CONFIDENCE_THRESHOLD):
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    probs = np.asarray(result["probs"], dtype=float)
    err = np.abs(pred - y)
    conf = probs.max(1)
    masks = {
        "all_incorrect": err >= 1,
        "adjacent_grade_error": err == 1,
        "two_grade_error": err == 2,
        ">=3_grade_error": err >= 3,
        "severe_error_>=2": err >= 2,
        f"high_conf_severe_error_p>={high_conf:.2f}": (err >= 2) & (conf >= high_conf),
    }
    return pd.DataFrame([
        {"category": k, "count": int(m.sum()), "fraction": float(m.mean()), "percent": float(100*m.mean())}
        for k, m in masks.items()
    ])


def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return (np.nan, np.nan)
    p = successes / total
    denom = 1.0 + z*z/total
    center = (p + z*z/(2*total)) / denom
    half = z * math.sqrt((p*(1-p)/total) + z*z/(4*total*total)) / denom
    return max(0.0, center-half), min(1.0, center+half)


def clinical_endpoint_metrics(result, threshold, endpoint_name):
    """Binary endpoint: positive if DR grade >= threshold.

    threshold=2 => referable DR (RDR)
    threshold=3 => vision-threatening DR (VTDR)
    """
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    probs = np.asarray(result["probs"], dtype=float)

    yt = y >= threshold
    yp = pred >= threshold
    positive_prob = probs[:, threshold:].sum(1)

    tp = int(np.sum(yt & yp))
    tn = int(np.sum(~yt & ~yp))
    fp = int(np.sum(~yt & yp))
    fn = int(np.sum(yt & ~yp))

    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    ppv = tp / max(tp + fp, 1)
    npv = tn / max(tn + fn, 1)
    acc = (tp + tn) / max(len(y), 1)
    sens_ci = wilson_interval(tp, tp + fn)
    spec_ci = wilson_interval(tn, tn + fp)

    try:
        auroc = float(roc_auc_score(yt.astype(int), positive_prob))
    except Exception:
        auroc = np.nan
    try:
        auprc = float(average_precision_score(yt.astype(int), positive_prob))
    except Exception:
        auprc = np.nan

    return {
        "endpoint": endpoint_name,
        "threshold": threshold,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Sensitivity": sens, "Specificity": spec, "PPV": ppv, "NPV": npv, "Accuracy": acc,
        "Sensitivity_CI_low": sens_ci[0], "Sensitivity_CI_high": sens_ci[1],
        "Specificity_CI_low": spec_ci[0], "Specificity_CI_high": spec_ci[1],
        "AUROC": auroc, "AUPRC": auprc,
    }


def clinical_screening_table(result):
    return pd.DataFrame([
        clinical_endpoint_metrics(result, 2, "Referable_DR"),
        clinical_endpoint_metrics(result, 3, "VTDR"),
    ])


def selective_prediction_curve(result, coverages=SELECTIVE_COVERAGES, uncertainty=SELECTIVE_UNCERTAINTY):
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    probs = np.asarray(result["probs"], dtype=float)
    if uncertainty == "entropy":
        u = predictive_entropy(probs)
    elif uncertainty == "1-maxprob":
        u = 1.0 - probs.max(1)
    else:
        raise ValueError("uncertainty must be 'entropy' or '1-maxprob'")

    order = np.argsort(u)  # lowest uncertainty first
    rows = []
    for cov in coverages:
        n_keep = max(2, int(round(len(y) * float(cov))))
        idx = order[:n_keep]
        sub = {
            "y": y[idx],
            "pred": pred[idx],
            "probs": probs[idx],
            "burden": np.asarray(result["burden"])[idx] if result.get("burden") is not None else None,
        }
        m = metrics_from_result(sub)
        rows.append({
            "coverage": float(cov),
            "retained_n": int(n_keep),
            "QWK": m["QWK"],
            "Macro-F1": m["Macro-F1"],
            "Accuracy": m["Accuracy"],
            "MAE": m["MAE"],
            "mean_uncertainty": float(u[idx].mean()),
        })
    return pd.DataFrame(rows)


def bootstrap_ci(result, repeats=BOOTSTRAP_REPEATS, seed=999):
    rng = np.random.default_rng(seed)
    n = len(result["y"])
    point = metrics_from_result(result)
    store = {k: [] for k in point}

    for _ in range(repeats):
        idx = rng.integers(0, n, size=n)
        boot = {
            "y": np.asarray(result["y"])[idx],
            "pred": np.asarray(result["pred"])[idx],
            "probs": np.asarray(result["probs"])[idx],
            "burden": np.asarray(result["burden"])[idx] if result.get("burden") is not None else None,
        }
        m = metrics_from_result(boot)
        for k, v in m.items():
            if np.isfinite(v):
                store[k].append(v)

    rows = []
    for k, v in point.items():
        vals = np.asarray(store[k], dtype=float)
        lo, hi = np.percentile(vals, [2.5, 97.5]) if len(vals) else (_ for _ in ()).throw(RuntimeError("Bootstrap produced no valid replicates."))
        rows.append({
            "metric": k, "value": v, "ci_low": lo, "ci_high": hi,
            "valid_bootstraps": len(vals),
        })
    return pd.DataFrame(rows)


def paired_bootstrap_difference(a, b, metric="QWK", repeats=BOOTSTRAP_REPEATS, seed=1234):
    assert list(a["ids"]) == list(b["ids"]), "Paired comparison requires identical ordered examples."
    y = np.asarray(a["y"])
    assert np.array_equal(y, np.asarray(b["y"])), "Ground-truth arrays differ."
    rng = np.random.default_rng(seed)
    diffs = []

    for _ in range(repeats):
        idx = rng.integers(0, len(y), len(y))
        aa = {
            "y": y[idx], "pred": np.asarray(a["pred"])[idx],
            "probs": np.asarray(a["probs"])[idx], "burden": None,
        }
        bb = {
            "y": y[idx], "pred": np.asarray(b["pred"])[idx],
            "probs": np.asarray(b["probs"])[idx], "burden": None,
        }
        ma, mb = metrics_from_result(aa)[metric], metrics_from_result(bb)[metric]
        if np.isfinite(ma) and np.isfinite(mb):
            diffs.append(ma - mb)

    diffs = np.asarray(diffs, dtype=float)
    return {
        "metric": metric,
        "delta_mean": float(diffs.mean()),
        "ci_low": float(np.percentile(diffs, 2.5)),
        "ci_high": float(np.percentile(diffs, 97.5)),
        "p_two_sided": float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))),
    }


def plot_reliability_diagram(result, name):
    tab = reliability_bins(result["probs"], result["y"], CALIBRATION_BINS)
    valid = tab["count"] > 0
    plt.figure(figsize=(5.5, 5.0))
    plt.plot([0, 1], [0, 1], "--", linewidth=1, label="Perfect calibration")
    plt.plot(tab.loc[valid, "confidence"], tab.loc[valid, "accuracy"], marker="o", label=name)
    plt.xlabel("Mean confidence")
    plt.ylabel("Empirical accuracy")
    plt.title(f"Reliability diagram — {name}")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.tight_layout()
    path = FIGURE_DIR / f"reliability_{name}.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    tab.to_csv(TABLE_DIR / f"reliability_bins_{name}.csv", index=False)
    return tab


In [15]:
# ============================== 13. Training / evaluation / checkpointing ==============================
def accumulation_denominator(step, n_batches, accum_steps=ACCUM_STEPS):
    remainder = n_batches % accum_steps
    if remainder and step >= n_batches - remainder:
        return remainder
    return accum_steps


def make_scheduler(optimizer):
    if WARMUP_EPOCHS > 0:
        warm = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.2, total_iters=WARMUP_EPOCHS)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS)
        )
        return torch.optim.lr_scheduler.SequentialLR(
            optimizer, [warm, cosine], milestones=[WARMUP_EPOCHS]
        )
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def train_epoch_enerdr(model, loader, optimizer):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss, n_seen, correct = 0.0, 0, 0
    component_sums = {}

    for step, (x1, x2, y, _) in enumerate(loader):
        x1 = x1.to(DEVICE, non_blocking=True)
        x2 = x2.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        # EnerDR runs in FP32 because RDEM gradient probes and energy safeguards are precision-sensitive.
        out1 = model(x1)
        out2 = model(x2) if model.switches.use_psic else None
        loss, comps, _ = compute_loss(out1, y, model.switches, out2)

        denom = accumulation_denominator(step, len(loader), ACCUM_STEPS)
        (loss / denom).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        bs = y.size(0)
        total_loss += float(loss.detach()) * bs
        n_seen += bs
        correct += int((out1["probs"].argmax(1) == y).sum().detach().cpu())
        for k, v in comps.items():
            component_sums[k] = component_sums.get(k, 0.0) + v * bs

    return (
        total_loss / max(n_seen, 1),
        {k: v / max(n_seen, 1) for k, v in component_sums.items()},
        correct / max(n_seen, 1),
    )


@torch.no_grad()
def evaluate_enerdr_objective(model, loader):
    model.eval()
    total, n = 0.0, 0
    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        out = model(x)
        loss, _, _ = compute_loss(out, y, model.switches, out2=None)
        total += float(loss.detach()) * y.size(0)
        n += y.size(0)
    return total / max(n, 1)


@torch.no_grad()
def predict_enerdr(model, loader):
    model.eval()
    ys, ids, probs, logits, burdens = [], [], [], [], []
    energy_traces, state_shifts = [], []
    acceptance_weighted_sum = None
    acceptance_n = 0

    for x, y, image_ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)
        bs = y.size(0)

        ys.extend(y.numpy().tolist())
        ids.extend(list(image_ids))
        probs.append(out["probs"].cpu().numpy())
        logits.append(out["logits"].cpu().numpy())
        burdens.append(out["burden"].cpu().numpy())

        if out.get("energy_trace"):
            trace = torch.stack(out["energy_trace"], dim=1).detach().cpu().numpy()
            energy_traces.append(trace)

        if out.get("state0") is not None and out.get("refined_state") is not None:
            shift = torch.linalg.vector_norm(
                (out["refined_state"] - out["state0"]).flatten(1), dim=1
            )
            state_shifts.append(shift.detach().cpu().numpy())

        if out.get("acceptance_rates"):
            step_rates = np.array([float(a.detach().cpu()) for a in out["acceptance_rates"]])
            if acceptance_weighted_sum is None:
                acceptance_weighted_sum = np.zeros_like(step_rates, dtype=float)
            acceptance_weighted_sum += step_rates * bs
            acceptance_n += bs

    probs = np.concatenate(probs)
    logits = np.concatenate(logits)
    burden = np.concatenate(burdens)
    y = np.asarray(ys, dtype=int)

    result = {
        "ids": ids,
        "y": y,
        "pred": probs.argmax(1),
        "probs": probs,
        "logits": logits,
        "burden": burden,
        "energy_trace": np.concatenate(energy_traces, axis=0) if energy_traces else None,
        "state_shift": np.concatenate(state_shifts, axis=0) if state_shifts else None,
        "mean_step_acceptance": (
            acceptance_weighted_sum / acceptance_n
            if acceptance_weighted_sum is not None and acceptance_n > 0
            else None
        ),
    }
    return result


def save_prediction_csv(result, path, seed_or_ensemble=None):
    probs = np.asarray(result["probs"], dtype=float)
    df = pd.DataFrame({
        "image_id": result["ids"],
        "y": result["y"],
        "pred": result["pred"],
        "burden": result.get("burden"),
        "confidence": probs.max(1),
        "uncertainty_1max": 1.0 - probs.max(1),
        "predictive_entropy": predictive_entropy(probs),
        "seed_or_ensemble": seed_or_ensemble if seed_or_ensemble is not None else Path(path).stem,
    })
    for k in range(NUM_CLASSES):
        df[f"p{k}"] = probs[:, k]
    df.to_csv(path, index=False)


def fit_enerdr(seed, switches=FULL_SWITCHES, tag="Full"):
    seed_everything(seed)
    train_loader, val_loader, test_loader, _ = make_enerdr_loaders(seed, include_external=False)
    model = EnerDRNetV3(switches).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer)

    ckpt = CHECKPOINT_DIR / f"EnerDR_{tag}_seed{seed}.pt"
    history = []
    best_qwk, best_epoch, wait = -np.inf, -1, 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, components, train_accuracy = train_epoch_enerdr(model, train_loader, optimizer)
        val_result = predict_enerdr(model, val_loader)
        val_metrics = metrics_from_result(val_result)
        val_loss = evaluate_enerdr_objective(model, val_loader)
        qwk = val_metrics["QWK"]
        scheduler.step()
        elapsed = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "epoch_seconds": elapsed,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            **{f"loss_{k}": v for k, v in components.items()},
        }
        history.append(row)
        print(
            f"[{tag} seed={seed}] epoch {epoch:03d} loss={train_loss:.4f} "
            f"train_acc={train_accuracy:.4f} val_loss={val_loss:.4f} "
            f"val_QWK={qwk:.4f} val_F1={val_metrics['Macro-F1']:.4f} time={elapsed:.1f}s"
        )

        if qwk > best_qwk + 1e-6:
            best_qwk, best_epoch, wait = qwk, epoch, 0
            torch.save({
                "model_state": model.state_dict(),
                "seed": seed,
                "epoch": epoch,
                "val_qwk": qwk,
                "switches": asdict(switches),
                "config": {**CONFIG, "NUM_PARTICLES": NUM_PARTICLES, "FLOW_STEPS": FLOW_STEPS, "RDEM_INTERACTION_WEIGHT": RDEM_INTERACTION_WEIGHT},
            }, ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(TABLE_DIR / f"history_{tag}_seed{seed}.csv", index=False)
    payload = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(payload["model_state"])
    model.eval()

    test_result = predict_enerdr(model, test_loader)
    save_prediction_csv(
        test_result,
        PREDICTION_DIR / f"APTOS_test_{tag}_seed{seed}.csv",
        seed_or_ensemble=f"seed_{seed}",
    )
    return model, ckpt, test_result, {
        "best_epoch": best_epoch,
        "best_val_qwk": best_qwk,
    }


def _dataset_prefix(dataset_name):
    return {
        "Messidor2_external": "Messidor_external",
        "IDRiD_external": "IDRiD_external",
    }.get(dataset_name, dataset_name.replace(" ", "_"))


def evaluate_enerdr_dataset(model, frame, seed, tag="Full", dataset_name="external"):
    result = predict_enerdr(model, _eval_loader(frame))
    prefix = _dataset_prefix(dataset_name)
    save_prediction_csv(
        result,
        PREDICTION_DIR / f"{prefix}_{tag}_seed{seed}.csv",
        seed_or_ensemble=f"seed_{seed}",
    )
    return result


def evaluate_enerdr_external(model, seed, tag="Full"):
    # Backward-compatible Messidor-2 wrapper.
    return evaluate_enerdr_dataset(
        model, MESSIDOR, seed, tag=tag, dataset_name="Messidor2_external"
    )


In [ ]:
# ============================== 14. Five-seed Full study; external sets opened only after APTOS training ==============================
def ensemble_results(results):
    if not results:
        raise ValueError("No results to ensemble")
    ids = list(results[0]["ids"])
    y = np.asarray(results[0]["y"])
    for r in results[1:]:
        assert ids == list(r["ids"])
        assert np.array_equal(y, np.asarray(r["y"]))

    probs = np.mean([r["probs"] for r in results], axis=0)
    burdens = (
        np.mean([r["burden"] for r in results], axis=0)
        if results[0].get("burden") is not None else None
    )

    # Ensemble diagnostic traces are averaged only when every seed supplies compatible arrays.
    energy_trace = None
    if all(r.get("energy_trace") is not None for r in results):
        shapes = {np.asarray(r["energy_trace"]).shape for r in results}
        if len(shapes) == 1:
            energy_trace = np.mean([r["energy_trace"] for r in results], axis=0)

    state_shift = None
    if all(r.get("state_shift") is not None for r in results):
        state_shift = np.mean([r["state_shift"] for r in results], axis=0)

    acceptance = None
    if all(r.get("mean_step_acceptance") is not None for r in results):
        acceptance = np.mean([r["mean_step_acceptance"] for r in results], axis=0)

    return {
        "ids": ids,
        "y": y,
        "pred": probs.argmax(1),
        "probs": probs,
        "burden": burdens,
        "energy_trace": energy_trace,
        "state_shift": state_shift,
        "mean_step_acceptance": acceptance,
    }


def load_enerdr_checkpoint(ckpt, switches=FULL_SWITCHES):
    model = EnerDRNetV3(switches).to(DEVICE)
    payload = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(payload["model_state"])
    model.eval()
    return model


def run_full_five_seed_study():
    internal_results, checkpoint_records, metric_rows = [], [], []

    # Phase 1: train/select ONLY on APTOS.
    for seed in FINAL_SEEDS:
        model, ckpt, test_result, info = fit_enerdr(seed, FULL_SWITCHES, "Full")
        internal_results.append(test_result)
        checkpoint_records.append((seed, ckpt))
        metric_rows.append({
            "seed": seed, "split": "APTOS_internal_test",
            **metrics_from_result(test_result), **info,
        })
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Phase 2: after all checkpoints are frozen, evaluate untouched external datasets.
    external_seed_results = {name: [] for name in EXTERNAL_FRAMES}
    for seed, ckpt in checkpoint_records:
        model = load_enerdr_checkpoint(ckpt, FULL_SWITCHES)
        for split_name, frame in EXTERNAL_FRAMES.items():
            ext = evaluate_enerdr_dataset(
                model, frame, seed, tag="Full", dataset_name=split_name
            )
            external_seed_results[split_name].append(ext)
            metric_rows.append({
                "seed": seed, "split": split_name,
                **metrics_from_result(ext),
            })
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metrics_df = pd.DataFrame(metric_rows)
    metrics_df.to_csv(TABLE_DIR / "EnerDR_Full_five_seed_metrics.csv", index=False)

    numeric_metrics = [
        c for c in metrics_df.columns
        if c not in {"seed", "split", "best_epoch"}
        and pd.api.types.is_numeric_dtype(metrics_df[c])
    ]
    summary_rows = []
    for split, g in metrics_df.groupby("split"):
        row = {"split": split}
        for c in numeric_metrics:
            row[c + "_mean"] = g[c].mean()
            row[c + "_sd"] = g[c].std(ddof=1)
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(TABLE_DIR / "EnerDR_Full_five_seed_summary.csv", index=False)

    internal_ensemble = ensemble_results(internal_results)
    save_prediction_csv(
        internal_ensemble,
        PREDICTION_DIR / "APTOS_test_Full_ensemble.csv",
        seed_or_ensemble="five_seed_ensemble",
    )
    bootstrap_ci(internal_ensemble).to_csv(
        TABLE_DIR / "APTOS_test_Full_ensemble_bootstrap_CI.csv", index=False
    )

    external_ensembles = {}
    for split_name, seed_results in external_seed_results.items():
        ens = ensemble_results(seed_results)
        external_ensembles[split_name] = ens
        prefix = _dataset_prefix(split_name)
        save_prediction_csv(
            ens,
            PREDICTION_DIR / f"{prefix}_Full_ensemble.csv",
            seed_or_ensemble="five_seed_ensemble",
        )
        bootstrap_ci(ens).to_csv(
            TABLE_DIR / f"{prefix}_Full_ensemble_bootstrap_CI.csv", index=False
        )

    print("\nFive-seed Full-model summary:\n", summary.to_string(index=False))
    print("\nAPTOS Full ensemble:", metrics_from_result(internal_ensemble))
    for split_name, ens in external_ensembles.items():
        print(f"\n{split_name} Full ensemble:", metrics_from_result(ens))

    return {
        "metrics": metrics_df,
        "summary": summary,
        "internal_results": internal_results,
        "internal_ensemble": internal_ensemble,
        "external_seed_results": external_seed_results,
        "external_ensembles": external_ensembles,
        # backward-compatible Messidor aliases
        "external_results": external_seed_results["Messidor2_external"],
        "external_ensemble": external_ensembles["Messidor2_external"],
        "idrid_results": external_seed_results["IDRiD_external"],
        "idrid_ensemble": external_ensembles["IDRiD_external"],
        "checkpoints": checkpoint_records,
    }


FULL_STUDY = run_full_five_seed_study()


[Full seed=17] epoch 001 loss=2.0489 train_acc=0.3679 val_loss=1.5190 val_QWK=0.0000 val_F1=0.1322 time=430.1s


In [ ]:
# ============================== 15. Genuine five-seed ablations ==============================
# Each ablation changes the actual forward computation. Messidor remains sealed by default.
def run_ablation_study():
    rows, ensembles = [], {}
    for name, switches in ABLATION_SWITCHES.items():
        print(f"\n===== Ablation: {name} =====")
        seed_results = []
        for seed in ABLATION_SEEDS:
            model, ckpt, test_result, info = fit_enerdr(seed, switches, name)
            seed_results.append(test_result)
            rows.append({"ablation": name, "seed": seed, "split": "APTOS_internal_test", **metrics_from_result(test_result), **info})

            if EVALUATE_EXTERNAL_FOR_ABLATIONS:
                ext = evaluate_enerdr_external(model, seed, name)
                rows.append({"ablation": name, "seed": seed, "split": "Messidor2_external", **metrics_from_result(ext)})

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        ens = ensemble_results(seed_results)
        ensembles[name] = ens
        save_prediction_csv(ens, PREDICTION_DIR / f"APTOS_test_{name}_ensemble.csv")
        bootstrap_ci(ens).to_csv(TABLE_DIR / f"APTOS_test_{name}_ensemble_bootstrap_CI.csv", index=False)

    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "ablation_multiseed_metrics.csv", index=False)
    metric_cols = [c for c in df.columns if c not in {"ablation", "seed", "split", "best_epoch"} and pd.api.types.is_numeric_dtype(df[c])]
    summary = df[df.split == "APTOS_internal_test"].groupby("ablation")[metric_cols].agg(["mean", "std"])
    summary.to_csv(TABLE_DIR / "ablation_multiseed_summary.csv")
    print("\nAblation summary:\n", summary)
    return {"metrics": df, "summary": summary, "ensembles": ensembles}


ABLATION_STUDY = run_ablation_study()


In [ ]:
# ============================== 16. Strong five-seed ImageNet-pretrained baselines ==============================
class BaselineClassifier(nn.Module):
    def __init__(self, name, pretrained=True):
        super().__init__()
        self.name = name
        if name == "ResNet50":
            weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
            self.model = models.resnet50(weights=weights)
            self.model.fc = nn.Linear(self.model.fc.in_features, NUM_CLASSES)
        elif name == "EfficientNetB3":
            weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = models.efficientnet_b3(weights=weights)
            self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, NUM_CLASSES)
        elif name == "ConvNeXtTiny":
            weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = models.convnext_tiny(weights=weights)
            self.model.classifier[2] = nn.Linear(self.model.classifier[2].in_features, NUM_CLASSES)
        elif name == "SwinTiny":
            weights = models.Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
            self.model = models.swin_t(weights=weights)
            self.model.head = nn.Linear(self.model.head.in_features, NUM_CLASSES)
        else:
            raise ValueError(f"Unknown baseline: {name}")

    def forward(self, x):
        return self.model(x)


def train_epoch_baseline(model, loader, optimizer):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total, n = 0.0, 0
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    for step, (x, y, _) in enumerate(loader):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = focal_loss(logits, y)
        denom = accumulation_denominator(step, len(loader), ACCUM_STEPS)
        scaler.scale(loss / denom).backward()
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        total += float(loss.detach()) * y.size(0)
        n += y.size(0)
    return total / max(n, 1)


@torch.no_grad()
def predict_baseline(model, loader):
    model.eval()
    ys, ids, pp, ll = [], [], [], []
    for x, y, image_ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        ys.extend(y.numpy().tolist())
        ids.extend(list(image_ids))
        pp.append(probs.cpu().numpy())
        ll.append(logits.cpu().numpy())
    probs = np.concatenate(pp)
    y = np.asarray(ys, dtype=int)
    return {"ids": ids, "y": y, "pred": probs.argmax(1), "probs": probs, "logits": np.concatenate(ll)}


def fit_baseline(name, seed):
    seed_everything(seed)
    train_loader, val_loader, test_loader, _ = make_baseline_loaders(seed, include_external=False)
    model = BaselineClassifier(name, BASELINE_PRETRAINED).to(DEVICE)
    lr = 3e-5 if BASELINE_PRETRAINED else 1e-4
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer)
    ckpt = CHECKPOINT_DIR / f"Baseline_{name}_seed{seed}.pt"
    best, best_epoch, wait, history = -np.inf, -1, 0, []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_epoch_baseline(model, train_loader, optimizer)
        val = predict_baseline(model, val_loader)
        vm = metrics_from_result(val)
        qwk = vm["QWK"]
        scheduler.step()
        history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in vm.items()}})
        print(f"[{name} seed={seed}] epoch={epoch:03d} loss={train_loss:.4f} val_QWK={qwk:.4f} time={time.time()-t0:.1f}s")
        if qwk > best + 1e-6:
            best, best_epoch, wait = qwk, epoch, 0
            torch.save({"model_state": model.state_dict(), "epoch": epoch, "val_qwk": qwk, "name": name, "seed": seed}, ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(TABLE_DIR / f"history_Baseline_{name}_seed{seed}.csv", index=False)
    if not ckpt.exists():
        raise RuntimeError(f"No checkpoint was created for {name}, seed {seed}.")
    payload = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(payload["model_state"])
    model.eval()
    test = predict_baseline(model, test_loader)
    save_prediction_csv(test, PREDICTION_DIR / f"APTOS_test_{name}_seed{seed}.csv")
    return model, ckpt, test, {"best_epoch": best_epoch, "best_val_qwk": best}


def load_baseline_checkpoint(name, ckpt):
    model = BaselineClassifier(name, BASELINE_PRETRAINED).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model_state"])
    model.eval()
    return model


def run_baselines():
    rows, ensembles = [], {}
    for name in BASELINE_NAMES:
        test_results, frozen_ckpts = [], []
        print(f"\n===== Baseline: {name} =====")
        for seed in BASELINE_SEEDS:
            model, ckpt, test, info = fit_baseline(name, seed)
            test_results.append(test)
            frozen_ckpts.append((seed, ckpt))
            rows.append({"model": name, "seed": seed, "split": "APTOS_internal_test", **metrics_from_result(test), **info})
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        external_results = {split_name: [] for split_name in EXTERNAL_FRAMES}
        for seed, ckpt in frozen_ckpts:
            model = load_baseline_checkpoint(name, ckpt)
            for split_name, frame in EXTERNAL_FRAMES.items():
                ext = predict_baseline(model, _eval_loader(frame))
                external_results[split_name].append(ext)
                prefix = _dataset_prefix(split_name)
                save_prediction_csv(ext, PREDICTION_DIR / f"{prefix}_{name}_seed{seed}.csv")
                rows.append({"model": name, "seed": seed, "split": split_name, **metrics_from_result(ext)})
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        int_ens = ensemble_results(test_results)
        ensembles[(name, "APTOS_internal_test")] = int_ens
        save_prediction_csv(int_ens, PREDICTION_DIR / f"APTOS_test_{name}_ensemble.csv")
        bootstrap_ci(int_ens).to_csv(TABLE_DIR / f"APTOS_test_{name}_ensemble_bootstrap_CI.csv", index=False)

        for split_name, seed_results in external_results.items():
            ens = ensemble_results(seed_results)
            ensembles[(name, split_name)] = ens
            prefix = _dataset_prefix(split_name)
            save_prediction_csv(ens, PREDICTION_DIR / f"{prefix}_{name}_ensemble.csv")
            bootstrap_ci(ens).to_csv(TABLE_DIR / f"{prefix}_{name}_ensemble_bootstrap_CI.csv", index=False)

    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "baseline_five_seed_metrics.csv", index=False)
    metric_cols = [c for c in df.columns if c not in {"model", "seed", "split", "best_epoch"} and pd.api.types.is_numeric_dtype(df[c])]
    summary = df.groupby(["model", "split"])[metric_cols].agg(["mean", "std"])
    summary.to_csv(TABLE_DIR / "baseline_five_seed_summary.csv")
    print("\nBaseline summary:\n", summary)
    return {"metrics": df, "summary": summary, "ensembles": ensembles}


BASELINE_STUDY = run_baselines()


In [ ]:
# ============================== 17. Statistical comparison + model accounting ==============================
def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        candidate = (m - rank) * p[idx]
        running = max(running, candidate)
        adjusted[idx] = min(1.0, running)
    return adjusted


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def save_parameter_table():
    rows = []
    m = EnerDRNetV3(FULL_SWITCHES)
    total, trainable = count_parameters(m)
    rows.append({"model": "EnerDR-Net Full", "total_params": total, "trainable_params": trainable,
                 "fp32_parameter_storage_MiB": total * 4 / 1024**2})
    del m
    for name in BASELINE_NAMES:
        m = BaselineClassifier(name, pretrained=False)
        total, trainable = count_parameters(m)
        rows.append({"model": name, "total_params": total, "trainable_params": trainable,
                     "fp32_parameter_storage_MiB": total * 4 / 1024**2})
        del m
    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "model_parameter_counts.csv", index=False)
    return df


def run_paired_comparisons():
    if FULL_STUDY is None or ABLATION_STUDY is None or BASELINE_STUDY is None:
        raise RuntimeError("Full study, ablations, and baselines must all be completed before paired comparisons.")

    comparisons = []
    metrics = ["QWK", "Macro-F1", "Balanced-Acc", "MAE"]
    datasets = {
        "APTOS_internal_test": FULL_STUDY["internal_ensemble"],
        "Messidor2_external": FULL_STUDY["external_ensembles"]["Messidor2_external"],
        "IDRiD_external": FULL_STUDY["external_ensembles"]["IDRiD_external"],
    }

    # Full vs ablations on APTOS.
    for label, candidate in ABLATION_STUDY["ensembles"].items():
        for metric in metrics:
            comparisons.append({"dataset": "APTOS_internal_test", "comparison": f"Full - Ablation:{label}",
                                **paired_bootstrap_difference(datasets["APTOS_internal_test"], candidate, metric=metric)})

    # Full vs every trained baseline on every dataset.
    for split_name, reference in datasets.items():
        for name in BASELINE_NAMES:
            key = (name, split_name)
            if key not in BASELINE_STUDY["ensembles"]:
                raise RuntimeError(f"Missing baseline ensemble: {key}")
            candidate = BASELINE_STUDY["ensembles"][key]
            for metric in metrics:
                comparisons.append({"dataset": split_name, "comparison": f"Full - Baseline:{name}",
                                    **paired_bootstrap_difference(reference, candidate, metric=metric)})

    df = pd.DataFrame(comparisons)
    df["p_holm"] = 1.0
    for (_, _), idx in df.groupby(["dataset", "metric"]).groups.items():
        idx = list(idx)
        df.loc[idx, "p_holm"] = holm_adjust(df.loc[idx, "p_two_sided"].values)
    df["significant_0.05_holm"] = df["p_holm"] < 0.05
    df.to_csv(TABLE_DIR / "paired_bootstrap_Full_vs_all.csv", index=False)
    return df


PARAMETER_TABLE = save_parameter_table()
PAIRED_COMPARISONS = run_paired_comparisons()
print("\nParameter counts:\n", PARAMETER_TABLE.to_string(index=False))
print("\nPaired bootstrap comparisons:\n", PAIRED_COMPARISONS.to_string(index=False))


In [ ]:
# ============================== 18. Native particle-evidence interpretability ==============================
@torch.no_grad()
def intrinsic_particle_evidence(model, x):
    """Pathological evidence map weighted by learned per-particle burden.

    This is stronger than simply averaging attention maps.
    """
    model.eval()
    out = model(x.to(DEVICE))
    pi = out["particle_maps"]                         # [B,N,H,W], each particle spatially normalized
    pb = out["particle_burden"].clamp_min(0.0)       # [B,N]
    evidence = (pi * pb[:, :, None, None]).sum(1) / (pb.sum(1)[:, None, None] + 1e-6)
    evidence = evidence / (evidence.amax(dim=(-1, -2), keepdim=True) + 1e-6)
    return evidence.cpu(), out


def plot_evidence_examples(model, frame, n=5):
    picks = []
    for grade in range(NUM_CLASSES):
        subset = frame[frame.grade == grade]
        if len(subset):
            picks.append(subset.iloc[0])
    picks = picks[:n]
    fig, axes = plt.subplots(len(picks), 3, figsize=(10, 3 * len(picks)))
    if len(picks) == 1:
        axes = axes[None, :]
    for i, row in enumerate(picks):
        with Image.open(row.cached_path) as im:
            im = im.convert("RGB")
            x = EVAL_TF(im).unsqueeze(0)
            ev, out = intrinsic_particle_evidence(model, x)
            axes[i, 0].imshow(im)
            axes[i, 0].set_title(f"True grade {row.grade}")
            axes[i, 1].imshow(ev[0], cmap="inferno")
            axes[i, 1].set_title("Burden-weighted particle evidence")
            axes[i, 2].imshow(im)
            axes[i, 2].imshow(ev[0], cmap="inferno", alpha=0.45)
            axes[i, 2].set_title(f"Pred={out['probs'].argmax(1).item()}  burden={out['burden'].item():.3f}")
            for ax in axes[i]:
                ax.axis("off")
    plt.tight_layout()
    path = FIGURE_DIR / "intrinsic_particle_evidence.png"
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    print("Saved:", path)


if RUN_INTERPRETABILITY:
    # Load the first final Full checkpoint; this block intentionally uses only a frozen model.
    seed = FINAL_SEEDS[0]
    ckpt = CHECKPOINT_DIR / f"EnerDR_Full_seed{seed}.pt"
    if not ckpt.exists():
        raise FileNotFoundError("Run the Full study first.")
    model = EnerDRNetV3(FULL_SWITCHES).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)["model_state"])
    plot_evidence_examples(model, APTOS_TEST, n=5)

In [ ]:
# ============================== 19. Core final reporting, plots, confusion matrices + manifest ==============================
def grade_burden_table_from_result(result):
    y = np.asarray(result["y"], dtype=int)
    burden = result.get("burden")
    if burden is None:
        return None
    df = pd.DataFrame({"y": y, "burden": np.asarray(burden, dtype=float)})
    return df.groupby("y")["burden"].agg(["count", "mean", "std", "median", "min", "max"])


def grade_burden_table(prediction_csv):
    df = pd.read_csv(prediction_csv)
    if "burden" not in df.columns or df["burden"].isna().all():
        return None
    return df.groupby("y")["burden"].agg(["count", "mean", "std", "median", "min", "max"])


def save_confusion_from_prediction_csv(prediction_csv, name):
    df = pd.read_csv(prediction_csv)
    cm = confusion_matrix(df.y, df.pred, labels=list(range(NUM_CLASSES)))
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{i}" for i in range(NUM_CLASSES)],
        columns=[f"pred_{i}" for i in range(NUM_CLASSES)],
    )
    cm_df.to_csv(TABLE_DIR / f"confusion_{name}.csv")

    row_sum = cm.sum(1, keepdims=True).clip(min=1)
    cm_norm = cm / row_sum
    plt.figure(figsize=(6, 5))
    plt.imshow(cm_norm, interpolation="nearest")
    plt.title(name.replace("_", " "))
    plt.xlabel("Predicted grade")
    plt.ylabel("True grade")
    plt.xticks(range(NUM_CLASSES))
    plt.yticks(range(NUM_CLASSES))
    plt.colorbar()
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            plt.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"confusion_{name}.png", dpi=220, bbox_inches="tight")
    plt.close()
    return cm


def plot_training_history(csv_path, name):
    df = pd.read_csv(csv_path)
    if df.empty:
        return

    if "val_QWK" in df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(df.epoch, df.val_QWK, label="Validation QWK")
        plt.xlabel("Epoch")
        plt.ylabel("QWK")
        plt.title(name)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"training_qwk_{name}.png", dpi=220, bbox_inches="tight")
        plt.close()

    if "train_loss" in df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(df.epoch, df.train_loss, label="Training loss")
        if "val_loss" in df.columns:
            plt.plot(df.epoch, df.val_loss, label="Validation objective")
        plt.xlabel("Epoch")
        plt.ylabel("Loss / objective")
        plt.title(name)
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"training_loss_{name}.png", dpi=220, bbox_inches="tight")
        plt.close()

    if "train_accuracy" in df.columns and "val_Accuracy" in df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(df.epoch, df.train_accuracy, label="Training accuracy")
        plt.plot(df.epoch, df.val_Accuracy, label="Validation accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title(name)
        plt.legend()
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"training_accuracy_{name}.png", dpi=220, bbox_inches="tight")
        plt.close()


def generate_final_reports():
    prediction_files = []

    canonical = [
        (PREDICTION_DIR / "APTOS_test_Full_ensemble.csv", "APTOS_Full_ensemble"),
        (PREDICTION_DIR / "Messidor_external_Full_ensemble.csv", "Messidor_Full_ensemble"),
        (PREDICTION_DIR / "IDRiD_external_Full_ensemble.csv", "IDRiD_Full_ensemble"),
    ]
    for p, label in canonical:
        if p.exists():
            prediction_files.append((p, label))

    for name in ABLATION_SWITCHES:
        p = PREDICTION_DIR / f"APTOS_test_{name}_ensemble.csv"
        if p.exists():
            prediction_files.append((p, f"APTOS_{name}_ensemble"))

    for name in BASELINE_NAMES:
        for prefix, label in [
            ("APTOS_test", "APTOS"),
            ("Messidor_external", "Messidor"),
            ("IDRiD_external", "IDRiD"),
        ]:
            p = PREDICTION_DIR / f"{prefix}_{name}_ensemble.csv"
            if p.exists():
                prediction_files.append((p, f"{label}_{name}_ensemble"))

    for p, name in prediction_files:
        save_confusion_from_prediction_csv(p, name)

    for p, label in canonical:
        if p.exists():
            burden = grade_burden_table(p)
            if burden is not None:
                burden.to_csv(TABLE_DIR / f"{label}_grade_burden.csv")

    for seed in FINAL_SEEDS:
        p = TABLE_DIR / f"history_Full_seed{seed}.csv"
        if p.exists():
            plot_training_history(p, f"Full_seed{seed}")

    return prediction_files


def final_manifest():
    manifest = {
        "study": "EnerDR-Net v4.0 complete APTOS development -> Messidor-2 + IDRiD frozen external testing",
        "architecture": "Encoder -> evidence particles -> RDEM -> safeguarded LREF -> DSV -> ODSA -> DEGD + PSIC",
        "aptos_csv": str(APTOS_CSV),
        "aptos_images": str(APTOS_IMAGE_DIR),
        "messidor_csv": str(MESSIDOR_CSV),
        "messidor_images": str(MESSIDOR_IMAGE_DIR),
        "idrid_root": str(IDRID_ROOT),
        "idrid_csv": str(IDRID_CSV) if IDRID_CSV is not None else "AUTO_DISCOVER",
        "external_policy": (
            "Messidor-2 and IDRiD excluded from optimization/checkpoint selection/early stopping; "
            "APTOS-selected checkpoints frozen before external evaluation."
        ),
        "duplicate_policy": (
            "Exact-image SHA256 audit enabled across APTOS splits and external datasets. "
            "Any train-to-held-out/external exact duplicate causes a RuntimeError."
        ),
        "split_seed": SPLIT_SEED,
        "training_seeds": FINAL_SEEDS,
        "ablation_seeds": ABLATION_SEEDS,
        "baseline_seeds": BASELINE_SEEDS,
        "sensitivity_seeds": SENSITIVITY_SEEDS,
        "ablation_names": list(ABLATION_SWITCHES.keys()),
        "baseline_names": ["ResNet50", "EfficientNetB3", "ConvNeXtTiny"],
        "preprocess_version": PREPROCESS_VERSION,
        "calibration_bins": CALIBRATION_BINS,
        "selective_uncertainty": SELECTIVE_UNCERTAINTY,
        "high_confidence_threshold": HIGH_CONFIDENCE_THRESHOLD,
        "active_particle_threshold": ACTIVE_PARTICLE_THRESHOLD,
        "torch": torch.__version__,
        "python": platform.python_version(),
        "device": str(DEVICE),
        "run_full": RUN_FULL_EXPERIMENT,
        "run_ablations": RUN_ABLATIONS,
        "run_baselines": RUN_BASELINES,
        "run_interpretability": RUN_INTERPRETABILITY,
        "run_paper_analyses": RUN_PAPER_ANALYSES,
        "run_efficiency_profile": RUN_EFFICIENCY_PROFILE,
        "run_sensitivity": RUN_SENSITIVITY,
    }
    (OUT_DIR / "study_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )
    return manifest


GENERATED_REPORTS = generate_final_reports()
print(json.dumps(final_manifest(), indent=2))
print(f"\nGenerated confusion reports for {len(GENERATED_REPORTS)} ensemble prediction files.")


In [ ]:
# ============================== 20. Paper-complete analyses generated from final predictions ==============================
def lref_diagnostics(result, full_mean_qwk, no_lref_mean_qwk):
    trace = result.get("energy_trace")
    if trace is None:
        raise RuntimeError("LREF energy trace is missing from the evaluated full model.")

    trace = np.asarray(trace, dtype=float)
    rows = []
    for t in range(trace.shape[1]):
        rows.append({"diagnostic": f"Mean_E{t}", "value": float(np.mean(trace[:, t]))})

    reduction = trace[:, 0] - trace[:, -1]
    rows.extend([
        {"diagnostic": "Mean_E0_minus_Efinal", "value": float(reduction.mean())},
        {"diagnostic": "Median_E0_minus_Efinal", "value": float(np.median(reduction))},
        {
            "diagnostic": "Energy_increase_violation_fraction",
            "value": float(np.mean(np.diff(trace, axis=1) > 2e-6)) if trace.shape[1] > 1 else 0.0,
        },
    ])

    acc = result.get("mean_step_acceptance")
    if acc is not None:
        acc = np.asarray(acc, dtype=float)
        rows.extend([
            {"diagnostic": "Accepted_refinement_steps", "value": float(acc.mean())},
            {"diagnostic": "Rejected_refinement_steps", "value": float(1.0 - acc.mean())},
        ])

    shift = result.get("state_shift")
    if shift is not None:
        rows.append({
            "diagnostic": "Mean_state_shift_L2",
            "value": float(np.asarray(shift, dtype=float).mean()),
        })

    if np.isfinite(full_mean_qwk):
        rows.append({"diagnostic": "QWK_with_LREF_five_seed_mean", "value": float(full_mean_qwk)})
    if np.isfinite(no_lref_mean_qwk):
        rows.append({"diagnostic": "QWK_without_LREF_five_seed_mean", "value": float(no_lref_mean_qwk)})
    if np.isfinite(full_mean_qwk) and np.isfinite(no_lref_mean_qwk):
        rows.append({
            "diagnostic": "Absolute_QWK_gain_LREF",
            "value": float(full_mean_qwk - no_lref_mean_qwk),
        })
    return pd.DataFrame(rows)


@torch.no_grad()
def structural_diagnostics_for_model(model, loader):
    model.eval()
    sym_max, diag_max, zero_inter_max = 0.0, 0.0, 0.0
    active_num, active_den = 0.0, 0
    latent_sum, latent_count = 0.0, 0
    overlap_sum, overlap_count = 0.0, 0
    interaction_burdens, ys = [], []

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        out = model(x)

        psi = out["psi"]
        B, N, _ = psi.shape
        eye = torch.eye(N, device=psi.device, dtype=torch.bool)[None]
        off = (~eye).expand(B, -1, -1)

        sym_max = max(sym_max, float((psi - psi.transpose(1, 2)).abs().max().cpu()))
        diag_max = max(diag_max, float(torch.diagonal(psi, dim1=1, dim2=2).abs().max().cpu()))

        zero_psi = torch.zeros_like(psi)
        zero_state_vec, _, _, _ = model.dsv(out["refined_state"], zero_psi)
        zero_inter_max = max(zero_inter_max, float(zero_state_vec[:, 1].abs().max().cpu()))

        activity = out["activity"]
        active_num += float((activity > ACTIVE_PARTICLE_THRESHOLD).sum().cpu())
        active_den += int(activity.numel())

        z = F.normalize(out["z"], p=2, dim=-1)
        zsim = torch.einsum("bnd,bmd->bnm", z, z).abs()
        latent_sum += float(zsim.masked_select(off).sum().cpu())
        latent_count += int(off.sum().cpu())

        p = F.normalize(out["particle_maps"].flatten(2), p=2, dim=-1)
        psim = torch.einsum("bnh,bmh->bnm", p, p)
        overlap_sum += float(psim.masked_select(off).sum().cpu())
        overlap_count += int(off.sum().cpu())

        inter = offdiag_mean(psi.abs()).detach().cpu().numpy()
        interaction_burdens.extend(inter.tolist())
        ys.extend(y.numpy().tolist())

    interaction_burdens = np.asarray(interaction_burdens, dtype=float)
    ys = np.asarray(ys, dtype=int)
    rho = float(pd.Series(interaction_burdens).corr(pd.Series(ys), method="spearman"))

    diag = {
        "interaction_symmetry_error_max": sym_max,
        "maximum_diagonal_interaction": diag_max,
        "DSV_interaction_under_zero_Psi_max": zero_inter_max,
        "active_particle_fraction": float(active_num / max(active_den, 1)),
        "mean_abs_latent_cosine_similarity": float(latent_sum / max(latent_count, 1)),
        "mean_attention_map_overlap": float(overlap_sum / max(overlap_count, 1)),
        "interaction_burden_grade_Spearman": rho,
    }

    grade_table = pd.DataFrame({"grade": ys, "interaction_burden": interaction_burdens}).groupby(
        "grade", as_index=False
    )["interaction_burden"].mean()
    return diag, grade_table

def aggregate_structural_diagnostics(checkpoints):
    diag_rows = []
    grade_tables = []
    loader = _eval_loader(APTOS_TEST)

    for seed, ckpt in checkpoints:
        model = load_enerdr_checkpoint(ckpt, FULL_SWITCHES)
        diag, grade = structural_diagnostics_for_model(model, loader)
        diag_rows.append({"seed": seed, **diag})
        grade = grade.copy()
        grade["seed"] = seed
        grade_tables.append(grade)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    per_seed = pd.DataFrame(diag_rows)
    summary = pd.DataFrame([
        {"diagnostic": c, "mean": per_seed[c].mean(), "sd": per_seed[c].std(ddof=1)}
        for c in per_seed.columns if c != "seed"
    ])
    grade_all = pd.concat(grade_tables, ignore_index=True)
    grade_summary = grade_all.groupby("grade")["interaction_burden"].agg(["mean", "std"]).reset_index()

    per_seed.to_csv(TABLE_DIR / "RDEM_structural_diagnostics_per_seed.csv", index=False)
    summary.to_csv(TABLE_DIR / "RDEM_structural_diagnostics_summary.csv", index=False)
    grade_summary.to_csv(TABLE_DIR / "RDEM_interaction_burden_by_grade.csv", index=False)
    return per_seed, summary, grade_summary


def degd_burden_center_alignment(checkpoints, aptos_ensemble):
    centers = []
    for seed, ckpt in checkpoints:
        model = load_enerdr_checkpoint(ckpt, FULL_SWITCHES)
        centers.append(model.decoder.centers().detach().cpu().numpy())
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    centers = np.stack(centers, axis=0)
    mean_centers = centers.mean(0)
    sd_centers = centers.std(0, ddof=1)

    y = np.asarray(aptos_ensemble["y"], dtype=int)
    burden = np.asarray(aptos_ensemble["burden"], dtype=float)
    rows = []
    for g in range(NUM_CLASSES):
        empirical = float(burden[y == g].mean())
        rows.append({
            "grade": g,
            "empirical_burden": empirical,
            "learned_center_mean": float(mean_centers[g]),
            "learned_center_sd": float(sd_centers[g]),
            "absolute_difference": abs(empirical - float(mean_centers[g])),
        })
    table = pd.DataFrame(rows)
    table.to_csv(TABLE_DIR / "DEGD_burden_center_alignment.csv", index=False)

    summary = pd.DataFrame([{
        "mean_absolute_burden_center_discrepancy": float(table["absolute_difference"].mean()),
        "strict_center_order_all_seeds": bool(np.all(np.diff(centers, axis=1) > 0)),
    }])
    summary.to_csv(TABLE_DIR / "DEGD_burden_center_alignment_summary.csv", index=False)
    return table, summary


def _per_class_table(result, dataset_name):
    y = np.asarray(result["y"], dtype=int)
    pred = np.asarray(result["pred"], dtype=int)
    f1s = f1_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    rec = recall_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    pre = precision_score(y, pred, labels=list(range(NUM_CLASSES)), average=None, zero_division=0)
    return pd.DataFrame({
        "dataset": dataset_name,
        "grade": np.arange(NUM_CLASSES),
        "precision": pre,
        "recall": rec,
        "F1": f1s,
        "support": [(y == g).sum() for g in range(NUM_CLASSES)],
    })


def run_complete_paper_analysis():
    if FULL_STUDY is None or ABLATION_STUDY is None or BASELINE_STUDY is None:
        raise RuntimeError("Full study, ablations, and baselines must be completed before result synthesis.")

    results = {
        "APTOS": FULL_STUDY["internal_ensemble"],
        "Messidor-2": FULL_STUDY["external_ensembles"]["Messidor2_external"],
        "IDRiD": FULL_STUDY["external_ensembles"]["IDRiD_external"],
    }

    # 1) Canonical ensemble performance + five-seed stability.
    five_seed = FULL_STUDY["metrics"]
    perf_rows = []
    for dataset_name, result in results.items():
        split_name = {
            "APTOS": "APTOS_internal_test",
            "Messidor-2": "Messidor2_external",
            "IDRiD": "IDRiD_external",
        }[dataset_name]
        m = metrics_from_result(result)
        q = five_seed[five_seed.split == split_name]["QWK"]
        perf_rows.append({
            "dataset": dataset_name,
            "Ensemble_QWK": m["QWK"],
            "Five_seed_QWK_mean": q.mean(),
            "Five_seed_QWK_SD": q.std(ddof=1),
            "Macro_F1": m["Macro-F1"],
            "Balanced_Accuracy": m["Balanced-Acc"],
            "Accuracy": m["Accuracy"],
            "Macro_AUROC": m["AUROC-OVR"],
            "Macro_AUPRC": m["AUPRC-OVR"],
            "MAE": m["MAE"],
            "ECE": m["ECE"],
            "MCE": m["MCE"],
            "Brier": m["Brier"],
            "NLL": m["NLL"],
            "Burden_Spearman": m["Burden-Spearman"],
        })
    master = pd.DataFrame(perf_rows)
    master.to_csv(TABLE_DIR / "paper_master_performance.csv", index=False)

    # Seed-level QWK table.
    qwk_seed = five_seed.pivot(index="seed", columns="split", values="QWK").reset_index()
    qwk_seed.to_csv(TABLE_DIR / "paper_five_seed_QWK.csv", index=False)

    # 2) Per-class, ordinal errors, calibration, clinical screening, selective prediction, failure analysis, burden.
    per_class_all, clinical_all, ordinal_all, failure_all, selective_all, burden_all = [], [], [], [], [], []
    for dataset_name, result in results.items():
        pc = _per_class_table(result, dataset_name)
        per_class_all.append(pc)

        ord_tab = ordinal_error_distribution(result)
        ord_tab.insert(0, "dataset", dataset_name)
        ordinal_all.append(ord_tab)

        clin = clinical_screening_table(result)
        clin.insert(0, "dataset", dataset_name)
        clinical_all.append(clin)

        fail = failure_case_summary(result)
        fail.insert(0, "dataset", dataset_name)
        failure_all.append(fail)

        sel = selective_prediction_curve(result)
        sel.insert(0, "dataset", dataset_name)
        selective_all.append(sel)

        burden = grade_burden_table_from_result(result)
        if burden is not None:
            b = burden.reset_index().rename(columns={"y": "grade"})
            b.insert(0, "dataset", dataset_name)
            burden_all.append(b)

        plot_reliability_diagram(result, dataset_name.replace("-", ""))

        bootstrap_ci(result).to_csv(
            TABLE_DIR / f"paper_bootstrap_CI_{dataset_name.replace('-', '')}.csv",
            index=False,
        )

    per_class = pd.concat(per_class_all, ignore_index=True)
    clinical = pd.concat(clinical_all, ignore_index=True)
    ordinal = pd.concat(ordinal_all, ignore_index=True)
    failure = pd.concat(failure_all, ignore_index=True)
    selective = pd.concat(selective_all, ignore_index=True)
    burden = pd.concat(burden_all, ignore_index=True)

    per_class.to_csv(TABLE_DIR / "paper_per_class_metrics.csv", index=False)
    clinical.to_csv(TABLE_DIR / "paper_clinical_RDR_VTDR.csv", index=False)
    ordinal.to_csv(TABLE_DIR / "paper_ordinal_error_distribution.csv", index=False)
    failure.to_csv(TABLE_DIR / "paper_failure_case_analysis.csv", index=False)
    selective.to_csv(TABLE_DIR / "paper_selective_prediction.csv", index=False)
    burden.to_csv(TABLE_DIR / "paper_disease_burden_progression.csv", index=False)

    # 3) Result-facing ablation and baseline comparison tables.
    ablation_paper = pd.DataFrame()
    if ABLATION_STUDY is not None:
        adf = ABLATION_STUDY["metrics"]
        rows = [{"configuration": "Full", "Five_seed_mean_QWK": five_seed.query("split == 'APTOS_internal_test'")["QWK"].mean()}]
        for name in ABLATION_SWITCHES:
            vals = adf[(adf.ablation == name) & (adf.split == "APTOS_internal_test")]["QWK"]
            if len(vals):
                rows.append({"configuration": name, "Five_seed_mean_QWK": vals.mean()})
        ablation_paper = pd.DataFrame(rows)
        full_value = float(ablation_paper.loc[ablation_paper.configuration == "Full", "Five_seed_mean_QWK"].iloc[0])
        ablation_paper["Drop_vs_Full"] = ablation_paper["Five_seed_mean_QWK"] - full_value
        ablation_paper.to_csv(TABLE_DIR / "paper_ablation_summary.csv", index=False)

    baseline_paper = pd.DataFrame()
    if BASELINE_STUDY is not None:
        rows = []
        for name in BASELINE_NAMES:
            int_res = BASELINE_STUDY["ensembles"].get((name, "APTOS_internal_test"))
            ext_res = BASELINE_STUDY["ensembles"].get((name, "Messidor2_external"))
            if int_res is not None and ext_res is not None:
                mi, me = metrics_from_result(int_res), metrics_from_result(ext_res)
                rows.append({
                    "model": name,
                    "APTOS_QWK": mi["QWK"],
                    "APTOS_MacroF1": mi["Macro-F1"],
                    "Messidor2_QWK": me["QWK"],
                    "Messidor2_MacroF1": me["Macro-F1"],
                    "APTOS_to_Messidor_QWK_drop": mi["QWK"] - me["QWK"],
                })
        fm_i = metrics_from_result(results["APTOS"])
        fm_e = metrics_from_result(results["Messidor-2"])
        rows.append({
            "model": "EnerDR-Net",
            "APTOS_QWK": fm_i["QWK"],
            "APTOS_MacroF1": fm_i["Macro-F1"],
            "Messidor2_QWK": fm_e["QWK"],
            "Messidor2_MacroF1": fm_e["Macro-F1"],
            "APTOS_to_Messidor_QWK_drop": fm_i["QWK"] - fm_e["QWK"],
        })
        baseline_paper = pd.DataFrame(rows)
        baseline_paper.to_csv(TABLE_DIR / "paper_baseline_generalization.csv", index=False)

    # 3) LREF energy diagnostics, using five-seed QWK means for with/without-LREF comparison.
    full_qwk_mean = five_seed[five_seed.split == "APTOS_internal_test"]["QWK"].mean()
    no_lref_qwk_mean = None
    if ABLATION_STUDY is not None:
        no_lref = ABLATION_STUDY["metrics"]
        no_lref_qwk_mean = no_lref[
            (no_lref.ablation == "No_LREF") &
            (no_lref.split == "APTOS_internal_test")
        ]["QWK"].mean()
    if no_lref_qwk_mean is None:
        raise RuntimeError("No-LREF five-seed QWK is unavailable.")
    lref = lref_diagnostics(
        results["APTOS"],
        full_mean_qwk=full_qwk_mean,
        no_lref_mean_qwk=no_lref_qwk_mean,
    )
    lref.to_csv(TABLE_DIR / "paper_LREF_energy_dynamics.csv", index=False)

    # 4) DEGD center/burden alignment.
    degd_table, degd_summary = degd_burden_center_alignment(
        FULL_STUDY["checkpoints"], results["APTOS"]
    )

    # 5) RDEM structural diagnostics.
    rdem_per_seed, rdem_summary, rdem_grade = aggregate_structural_diagnostics(
        FULL_STUDY["checkpoints"]
    )

    # 6) Generalization gaps.
    apt_qwk = float(master.loc[master.dataset == "APTOS", "Ensemble_QWK"].iloc[0])
    gen = pd.DataFrame([
        {
            "transfer": "APTOS_to_Messidor2",
            "internal_QWK": apt_qwk,
            "external_QWK": float(master.loc[master.dataset == "Messidor-2", "Ensemble_QWK"].iloc[0]),
        },
        {
            "transfer": "APTOS_to_IDRiD",
            "internal_QWK": apt_qwk,
            "external_QWK": float(master.loc[master.dataset == "IDRiD", "Ensemble_QWK"].iloc[0]),
        },
    ])
    gen["QWK_drop"] = gen["internal_QWK"] - gen["external_QWK"]
    gen["QWK_retention"] = gen["external_QWK"] / gen["internal_QWK"]
    gen.to_csv(TABLE_DIR / "paper_cross_dataset_generalization.csv", index=False)


    # 7) Measured training-time and convergence summary from the actual history CSVs.
    timing_rows = []
    convergence_rows = []
    for seed in FINAL_SEEDS:
        hp = TABLE_DIR / f"history_Full_seed{seed}.csv"
        if not hp.exists():
            continue
        h = pd.read_csv(hp)
        if "epoch_seconds" in h.columns and len(h):
            timing_rows.append({
                "seed": seed,
                "epochs_run": len(h),
                "mean_epoch_seconds": float(h["epoch_seconds"].mean()),
                "median_epoch_seconds": float(h["epoch_seconds"].median()),
                "best_val_QWK": float(h["val_QWK"].max()),
            })
        if len(h):
            convergence_rows.append({
                "seed": seed,
                "train_loss_start": float(h["train_loss"].iloc[0]),
                "train_loss_end": float(h["train_loss"].iloc[-1]),
                "val_loss_start": float(h["val_loss"].iloc[0]),
                "val_loss_end": float(h["val_loss"].iloc[-1]),
                "train_accuracy_start": float(h["train_accuracy"].iloc[0]),
                "train_accuracy_end": float(h["train_accuracy"].iloc[-1]),
                "val_accuracy_start": float(h["val_Accuracy"].iloc[0]),
                "val_accuracy_end": float(h["val_Accuracy"].iloc[-1]),
                "val_QWK_start": float(h["val_QWK"].iloc[0]),
                "val_QWK_best": float(h["val_QWK"].max()),
            })
    timing = pd.DataFrame(timing_rows)
    convergence = pd.DataFrame(convergence_rows)
    if len(timing):
        timing.to_csv(TABLE_DIR / "paper_training_time_per_epoch.csv", index=False)
    if len(convergence):
        convergence.to_csv(TABLE_DIR / "paper_training_convergence_summary.csv", index=False)

    print("\nPaper master performance:\n", master.round(4).to_string(index=False))
    print("\nClinical endpoints:\n", clinical.round(4).to_string(index=False))
    print("\nLREF diagnostics:\n", lref.round(6).to_string(index=False))
    print("\nDEGD alignment:\n", degd_table.round(4).to_string(index=False))
    print("\nRDEM diagnostics:\n", rdem_summary.round(6).to_string(index=False))

    return {
        "results": results,
        "master": master,
        "qwk_seed": qwk_seed,
        "per_class": per_class,
        "clinical": clinical,
        "ordinal": ordinal,
        "failure": failure,
        "selective": selective,
        "burden": burden,
        "ablation_paper": ablation_paper,
        "baseline_paper": baseline_paper,
        "lref": lref,
        "degd_alignment": degd_table,
        "degd_summary": degd_summary,
        "rdem_per_seed": rdem_per_seed,
        "rdem_summary": rdem_summary,
        "rdem_grade": rdem_grade,
        "generalization": gen,
        "training_timing": timing,
        "convergence": convergence,
    }


PAPER_ANALYSIS = run_complete_paper_analysis()


In [ ]:
# ============================== 21. Hyperparameter sensitivity analysis ==============================
# This block reproduces the paper sensitivity experiments directly from training runs.
# It is disabled by default because, with five seeds, it is computationally expensive.

@contextmanager
def temporary_globals(**changes):
    old = {}
    g = globals()
    try:
        for k, v in changes.items():
            old[k] = g[k]
            g[k] = v
        yield
    finally:
        for k, v in old.items():
            g[k] = v


def _run_sensitivity_training(tag, switches=FULL_SWITCHES, **global_changes):
    seed_results = []
    with temporary_globals(**global_changes):
        for seed in SENSITIVITY_SEEDS:
            model, ckpt, test_result, _ = fit_enerdr(seed, switches=switches, tag=tag)
            seed_results.append(test_result)
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return ensemble_results(seed_results)


def _reuse_or_run_sensitivity(kind, value):
    # Reuse exact experiments already completed when mathematically identical.
    if kind == "N" and value == NUM_PARTICLES and FULL_STUDY is not None:
        return FULL_STUDY["internal_ensemble"]

    if kind == "T":
        if value == FLOW_STEPS and FULL_STUDY is not None:
            return FULL_STUDY["internal_ensemble"]
        if value == 0 and ABLATION_STUDY is not None:
            return ABLATION_STUDY["ensembles"]["No_LREF"]
        switches = FULL_SWITCHES if value > 0 else ModelSwitches(use_lref=False)
        return _run_sensitivity_training(
            f"Sensitivity_T{value}",
            switches=switches,
            FLOW_STEPS=int(value),
        )

    if kind == "lambda_int":
        if abs(float(value) - RDEM_INTERACTION_WEIGHT) < 1e-12 and FULL_STUDY is not None:
            return FULL_STUDY["internal_ensemble"]
        if float(value) == 0.0 and ABLATION_STUDY is not None:
            return ABLATION_STUDY["ensembles"]["No_Interaction"]
        switches = FULL_SWITCHES if float(value) > 0 else ModelSwitches(use_interaction=False)
        return _run_sensitivity_training(
            f"Sensitivity_lambda{str(value).replace('.', 'p')}",
            switches=switches,
            RDEM_INTERACTION_WEIGHT=float(value),
        )

    if kind == "N":
        return _run_sensitivity_training(
            f"Sensitivity_N{value}",
            switches=FULL_SWITCHES,
            NUM_PARTICLES=int(value),
        )

    raise ValueError((kind, value))


def run_hyperparameter_sensitivity():
    grids = {
        "N": [8, 16, 24, 32, 48],
        "T": [0, 1, 2, 3, 4, 5],
        "lambda_int": [0.00, 0.05, 0.10, 0.15, 0.20, 0.25],
    }

    rows = []
    for kind, values in grids.items():
        for value in values:
            print(f"\n===== Sensitivity {kind}={value} =====")
            result = _reuse_or_run_sensitivity(kind, value)
            m = metrics_from_result(result)
            rows.append({
                "hyperparameter": kind,
                "value": value,
                "QWK": m["QWK"],
                "Macro-F1": m["Macro-F1"],
                "Balanced-Acc": m["Balanced-Acc"],
                "Accuracy": m["Accuracy"],
                "MAE": m["MAE"],
            })

    df = pd.DataFrame(rows)
    df.to_csv(TABLE_DIR / "paper_hyperparameter_sensitivity.csv", index=False)

    # One figure per hyperparameter.
    for kind, g in df.groupby("hyperparameter"):
        g = g.sort_values("value")
        plt.figure(figsize=(6, 4))
        plt.plot(g["value"], g["QWK"], marker="o", label="QWK")
        plt.plot(g["value"], g["Macro-F1"], marker="s", label="Macro-F1")
        plt.xlabel(kind)
        plt.ylabel("Score")
        plt.title(f"Sensitivity to {kind}")
        plt.grid(alpha=0.2)
        plt.legend()
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"sensitivity_{kind}.png", dpi=220, bbox_inches="tight")
        plt.close()

    print(df.round(4).to_string(index=False))
    return df


SENSITIVITY_RESULTS = run_hyperparameter_sensitivity()


In [ ]:
# ============================== 22. Computational efficiency profiling ==============================
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def _profile_flops(model, x):
    from torch.profiler import profile, ProfilerActivity
    activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA] if torch.cuda.is_available() else [ProfilerActivity.CPU]
    try:
        with profile(activities=activities, with_flops=True, record_shapes=False) as prof:
            _ = model(x)
            _sync()
        flops = sum((evt.flops or 0) for evt in prof.key_averages())
    except Exception as e:
        raise RuntimeError(f"FLOP profiling failed: {e}") from e
    if not np.isfinite(flops) or flops <= 0:
        raise RuntimeError("FLOP profiler returned a non-positive result.")
    return float(flops / 1e9)


def _latency_and_memory(model, input_shape=(1, 3, IMAGE_SIZE, IMAGE_SIZE), warmup=20, reps=100):
    if not torch.cuda.is_available():
        raise RuntimeError("Efficiency profiling requires CUDA so latency and GPU-memory results are complete and comparable.")
    model = model.to(DEVICE).eval()
    x = torch.randn(*input_shape, device=DEVICE)
    for _ in range(warmup):
        with torch.no_grad():
            _ = model(x)
    _sync()

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(x)
    _sync()
    peak_mem_gb = float(torch.cuda.max_memory_allocated() / 1024**3)

    times = []
    for _ in range(reps):
        _sync(); t0 = time.perf_counter()
        with torch.no_grad():
            _ = model(x)
        _sync(); times.append((time.perf_counter() - t0) * 1000.0)
    latency_ms = float(np.median(times))
    throughput = float(input_shape[0] * 1000.0 / latency_ms)
    gflops = _profile_flops(model, x)
    return latency_ms, throughput, peak_mem_gb, gflops


def _training_peak_memory(model, model_name):
    if not torch.cuda.is_available():
        raise RuntimeError("Training-memory profiling requires CUDA.")
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = model.to(DEVICE).train()
    x1 = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    y = torch.arange(BATCH_SIZE, device=DEVICE) % NUM_CLASSES
    model.zero_grad(set_to_none=True)
    if model_name == "EnerDR-Net":
        x2 = torch.randn_like(x1)
        out1 = model(x1); out2 = model(x2)
        loss, _, _ = compute_loss(out1, y, FULL_SWITCHES, out2)
    else:
        loss = focal_loss(model(x1), y)
    loss.backward(); _sync()
    peak = float(torch.cuda.max_memory_allocated() / 1024**3)
    model.zero_grad(set_to_none=True)
    return peak


def profile_all_models():
    if FULL_STUDY is None or not FULL_STUDY.get("checkpoints"):
        raise RuntimeError("Full-model checkpoint is required for efficiency profiling.")
    rows = []
    model_specs = [("EnerDR-Net", lambda: load_enerdr_checkpoint(FULL_STUDY["checkpoints"][0][1], FULL_SWITCHES))]
    model_specs += [(name, lambda n=name: BaselineClassifier(n, pretrained=False).to(DEVICE)) for name in BASELINE_NAMES]

    for name, builder in model_specs:
        model = builder()
        total, trainable = count_parameters(model)
        lat, thr, mem, flops = _latency_and_memory(model)
        train_mem = _training_peak_memory(model, name)
        rows.append({"model": name, "params": total, "trainable_params": trainable,
                     "GFLOPs_per_image": flops, "latency_ms_per_image": lat,
                     "throughput_images_per_s": thr, "peak_inference_memory_GB": mem,
                     "peak_training_memory_GB": train_mem})
        del model; gc.collect(); torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df["hardware"] = torch.cuda.get_device_name(0)
    df["input_resolution"] = f"{IMAGE_SIZE}x{IMAGE_SIZE}"
    if not np.isfinite(df.select_dtypes(include=[np.number]).to_numpy()).all():
        raise RuntimeError("Efficiency profile contains a missing or non-finite numeric value.")
    df.to_csv(TABLE_DIR / "paper_efficiency_profile.csv", index=False)
    print("\nEfficiency profile:\n", df.round(4).to_string(index=False))
    return df


EFFICIENCY_PROFILE = profile_all_models()



## Paper-facing outputs generated by this notebook

After a complete run, use the generated files under `tables/`, `figures/`, and `predictions/` as the manuscript source of truth.

Major outputs include:

- `paper_master_performance.csv`: APTOS, Messidor-2 and IDRiD ensemble performance plus five-seed QWK stability.
- `paper_per_class_metrics.csv`, confusion matrices and `paper_ordinal_error_distribution.csv`.
- `paper_clinical_RDR_VTDR.csv`: RDR/VTDR sensitivity, specificity, PPV, NPV, accuracy, Wilson CIs, AUROC and AUPRC.
- `reliability_bins_*.csv` and reliability-diagram figures; ECE, MCE, Brier and NLL are computed from the same saved probabilities.
- `paper_selective_prediction.csv` and `paper_failure_case_analysis.csv`.
- `paper_disease_burden_progression.csv`, `DEGD_burden_center_alignment.csv`.
- `paper_LREF_energy_dynamics.csv`.
- `RDEM_structural_diagnostics_summary.csv` and `RDEM_interaction_burden_by_grade.csv`.
- `paper_ablation_summary.csv`, `paper_baseline_generalization.csv`, and paired-bootstrap statistics.
- `paper_cross_dataset_generalization.csv`.
- `paper_training_time_per_epoch.csv` and `paper_training_convergence_summary.csv`.
- `paper_efficiency_profile.csv`.
- `paper_hyperparameter_sensitivity.csv` when `RUN_SENSITIVITY=True`.
- `complete_run_audit.json`, which verifies that required run-derived outputs are present and numerically finite.

**Literature/SOTA comparisons are intentionally not generated as experimental output.** Published-method values must be taken
from the cited papers and entered into the manuscript with the corresponding evaluation protocol. They should never be made
to look like measurements produced by this notebook.


## Additional validation and robustness experiments

This section adds the following experiments **without hard-coded target results**:

1. Stratified 5-fold APTOS cross-validation with fold-trained frozen evaluation on full Messidor-2 and IDRiD.
2. Validation-fitted temperature scaling and before/after calibration analysis.
3. Grad-CAM, intrinsic particle-attention visualization, occlusion faithfulness, and optional IDRiD lesion-mask localization.
4. Cross-dataset feature-space domain-shift/applicability analysis.
5. Preprocessing ablation.
6. Augmentation ablation.
7. Validation-selected operating-point analysis for RDR and VTDR.

All reported values are computed from model outputs generated at runtime. No manuscript target values are embedded in the implementation.


In [ ]:
# ============================== 23. New paper experiments: configuration + generic helpers ==============================
from sklearn.model_selection import StratifiedKFold

# Master switch. Set individual switches False if you want to run experiments separately.
RUN_NEW_PAPER_EXPERIMENTS = True
RUN_STRATIFIED_5FOLD_CV = True
RUN_TEMPERATURE_CALIBRATION = True
RUN_XAI_LOCALIZATION = True
RUN_DOMAIN_SHIFT = True
RUN_PREPROCESSING_ABLATION = True
RUN_AUGMENTATION_ABLATION = True
RUN_OPERATING_POINTS = True

CV_FOLDS = 5
CV_INNER_VAL_FRAC = 0.15
CV_SEEDS = FINAL_SEEDS[:CV_FOLDS]
NEW_EXPERIMENT_SEEDS = FINAL_SEEDS
XAI_MAX_SAMPLES = 100
XAI_TOP_FRACTION = 0.20
DOMAIN_BATCH_SIZE = max(BATCH_SIZE, 8)
RDR_HIGH_SENSITIVITY_TARGET = 0.95
VTDR_HIGH_SENSITIVITY_TARGET = 0.90

NEW_TABLE_DIR = TABLE_DIR / "extended_experiments"
NEW_FIGURE_DIR = FIGURE_DIR / "extended_experiments"
NEW_PRED_DIR = PREDICTION_DIR / "extended_experiments"
NEW_CACHE_DIR = CACHE_DIR / "extended_experiments"
NEW_CKPT_DIR = CHECKPOINT_DIR / "extended_experiments"
for _p in [NEW_TABLE_DIR, NEW_FIGURE_DIR, NEW_PRED_DIR, NEW_CACHE_DIR, NEW_CKPT_DIR]:
    _p.mkdir(parents=True, exist_ok=True)



# Reconstruct the complete preprocessed APTOS pool for cross-validation.
# The canonical notebook caches train/val/test separately, while outer CV must use all 3,662 images.
APTOS_ALL_CACHED = pd.concat([APTOS_TRAIN, APTOS_VAL, APTOS_TEST], ignore_index=True)
APTOS_ALL_CACHED = APTOS_ALL_CACHED.drop_duplicates(subset=["image_id"]).reset_index(drop=True)
assert len(APTOS_ALL_CACHED) == len(APTOS), "APTOS cached pool does not match the audited full dataset."

def _safe_tag(x):
    return str(x).replace(" ", "_").replace("/", "_").replace("\\", "_")


def _result_with_probs(base_result, probs):
    probs = np.asarray(probs, dtype=float)
    out = dict(base_result)
    out["probs"] = probs
    out["pred"] = probs.argmax(1)
    return out


def _summary_mean_sd(df, group_col, metric_cols):
    rows = []
    for key, g in df.groupby(group_col):
        row = {group_col: key, "n_runs": len(g)}
        for c in metric_cols:
            row[f"{c}_mean"] = float(g[c].mean())
            row[f"{c}_sd"] = float(g[c].std(ddof=1)) if len(g) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def _full_checkpoint_records():
    if FULL_STUDY is not None and FULL_STUDY.get("checkpoints"):
        return [(int(seed), Path(path)) for seed, path in FULL_STUDY["checkpoints"]]
    rec = []
    for seed in FINAL_SEEDS:
        p = CHECKPOINT_DIR / f"EnerDR_Full_seed{seed}.pt"
        if p.exists():
            rec.append((seed, p))
    if len(rec) != len(FINAL_SEEDS):
        raise RuntimeError(
            "Full-model checkpoints are required. Run the Full five-seed study first or keep RUN_FULL_EXPERIMENT=True."
        )
    return rec


class ExperimentRetinaDataset(Dataset):
    """Dataset used only by the added experiments.

    aug_mode:
      - 'none': no stochastic augmentation
      - 'geom': shared geometric augmentation only
      - 'geom_photo': shared geometry plus independent photometric views
    """
    def __init__(self, frame, train=False, two_views=True, path_col="cached_path", aug_mode="geom_photo"):
        self.frame = frame.reset_index(drop=True)
        self.train = bool(train)
        self.two_views = bool(two_views)
        self.path_col = path_col
        self.aug_mode = aug_mode
        if path_col not in self.frame.columns:
            raise KeyError(f"Missing image-path column: {path_col}")
        if aug_mode not in {"none", "geom", "geom_photo"}:
            raise ValueError(f"Unknown aug_mode={aug_mode}")

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row[self.path_col]) as im:
            im = im.convert("RGB")
            if not self.train:
                x = TO_TENSOR_NORM(im)
                return x, int(row.grade), str(row.image_id)

            if self.aug_mode == "none":
                g = im
                x1 = TO_TENSOR_NORM(g)
                x2 = x1.clone() if self.two_views else None
            elif self.aug_mode == "geom":
                g = GEOM_TF(im)
                x1 = TO_TENSOR_NORM(g)
                x2 = x1.clone() if self.two_views else None
            else:
                g = GEOM_TF(im)
                x1 = TO_TENSOR_NORM(PHOTO_TF(g))
                x2 = TO_TENSOR_NORM(PHOTO_TF(g)) if self.two_views else None

            if self.two_views:
                return x1, x2, int(row.grade), str(row.image_id)
            return x1, int(row.grade), str(row.image_id)


def _experiment_eval_loader(frame, path_col="cached_path"):
    return DataLoader(
        ExperimentRetinaDataset(frame, train=False, path_col=path_col),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    )


def _experiment_train_loader(frame, seed, path_col="cached_path", aug_mode="geom_photo"):
    return DataLoader(
        ExperimentRetinaDataset(frame, train=True, two_views=True, path_col=path_col, aug_mode=aug_mode),
        batch_size=BATCH_SIZE,
        sampler=make_sampler(frame, seed),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker if NUM_WORKERS > 0 else None,
    )


def fit_enerdr_frames(train_df, val_df, test_df, seed, tag,
                      switches=FULL_SWITCHES, path_col="cached_path", aug_mode="geom_photo",
                      reuse_checkpoint=True):
    """Train EnerDR-Net on explicitly supplied partitions.

    This prevents the new CV/ablation experiments from mutating the canonical global split.
    """
    seed_everything(int(seed))
    train_loader = _experiment_train_loader(train_df, seed, path_col=path_col, aug_mode=aug_mode)
    val_loader = _experiment_eval_loader(val_df, path_col=path_col)
    test_loader = _experiment_eval_loader(test_df, path_col=path_col)

    model = EnerDRNetV3(switches).to(DEVICE)
    ckpt = NEW_CKPT_DIR / f"EnerDR_{_safe_tag(tag)}_seed{seed}.pt"
    history_path = NEW_TABLE_DIR / f"history_{_safe_tag(tag)}_seed{seed}.csv"

    if reuse_checkpoint and ckpt.exists():
        payload = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(payload["model_state"])
        model.eval()
        test_result = predict_enerdr(model, test_loader)
        return model, ckpt, test_result, {
            "best_epoch": int(payload.get("epoch", -1)),
            "best_val_qwk": float(payload["val_qwk"]),
            "reused_checkpoint": True,
        }

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = make_scheduler(optimizer)
    history = []
    best_qwk, best_epoch, wait = -np.inf, -1, 0

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss, components, train_accuracy = train_epoch_enerdr(model, train_loader, optimizer)
        val_result = predict_enerdr(model, val_loader)
        val_metrics = metrics_from_result(val_result)
        val_loss = evaluate_enerdr_objective(model, val_loader)
        qwk = val_metrics["QWK"]
        scheduler.step()
        elapsed = time.time() - t0
        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_accuracy,
            "val_loss": val_loss, "epoch_seconds": elapsed,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            **{f"loss_{k}": v for k, v in components.items()},
        })
        print(
            f"[{tag} seed={seed}] epoch {epoch:03d} loss={train_loss:.4f} "
            f"val_QWK={qwk:.4f} val_F1={val_metrics['Macro-F1']:.4f} time={elapsed:.1f}s"
        )

        if qwk > best_qwk + 1e-6:
            best_qwk, best_epoch, wait = qwk, epoch, 0
            torch.save({
                "model_state": model.state_dict(), "seed": int(seed), "epoch": int(epoch),
                "val_qwk": float(qwk), "switches": asdict(switches),
                "tag": str(tag), "path_col": path_col, "aug_mode": aug_mode,
                "train_ids_sha256": hashlib.sha256("|".join(train_df.image_id.astype(str)).encode()).hexdigest(),
                "val_ids_sha256": hashlib.sha256("|".join(val_df.image_id.astype(str)).encode()).hexdigest(),
                "test_ids_sha256": hashlib.sha256("|".join(test_df.image_id.astype(str)).encode()).hexdigest(),
            }, ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(history_path, index=False)
    payload = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(payload["model_state"])
    model.eval()
    test_result = predict_enerdr(model, test_loader)
    save_prediction_csv(
        test_result,
        NEW_PRED_DIR / f"{_safe_tag(tag)}_test_seed{seed}.csv",
        seed_or_ensemble=f"seed_{seed}",
    )
    return model, ckpt, test_result, {
        "best_epoch": best_epoch, "best_val_qwk": best_qwk, "reused_checkpoint": False,
    }


def evaluate_frame(model, frame, dataset_name, tag, seed, path_col="cached_path"):
    result = predict_enerdr(model, _experiment_eval_loader(frame, path_col=path_col))
    save_prediction_csv(
        result,
        NEW_PRED_DIR / f"{_safe_tag(dataset_name)}_{_safe_tag(tag)}_seed{seed}.csv",
        seed_or_ensemble=f"seed_{seed}",
    )
    return result


In [ ]:
# ============================== 24. Stratified 5-fold CV + frozen external evaluation ==============================
def run_stratified_5fold_external_study():
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SPLIT_SEED)
    y_all = APTOS_ALL_CACHED.grade.to_numpy(int)
    fold_rows = []
    fold_results = {"APTOS": [], "Messidor-2": [], "IDRiD": []}
    records = []

    for fold, (outer_train_idx, test_idx) in enumerate(skf.split(np.zeros(len(APTOS_ALL_CACHED)), y_all), start=1):
        seed = int(CV_SEEDS[fold - 1])
        outer_train = APTOS_ALL_CACHED.iloc[outer_train_idx].reset_index(drop=True)
        test_df = APTOS_ALL_CACHED.iloc[test_idx].reset_index(drop=True)
        train_df, val_df = train_test_split(
            outer_train, test_size=CV_INNER_VAL_FRAC,
            stratify=outer_train["grade"], random_state=SPLIT_SEED + fold,
        )
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)

        # Audit exact image-id separation for every fold.
        assert set(train_df.image_id).isdisjoint(set(val_df.image_id))
        assert set(train_df.image_id).isdisjoint(set(test_df.image_id))
        assert set(val_df.image_id).isdisjoint(set(test_df.image_id))

        tag = f"CV_fold{fold}"
        model, ckpt, aptos_result, info = fit_enerdr_frames(
            train_df, val_df, test_df, seed=seed, tag=tag,
            switches=FULL_SWITCHES, path_col="cached_path", aug_mode="geom_photo",
        )
        records.append((fold, seed, ckpt))
        fold_results["APTOS"].append(aptos_result)
        ma = metrics_from_result(aptos_result)
        fold_rows.append({"fold": fold, "seed": seed, "dataset": "APTOS", **ma, **info})

        # External labels are not used for fitting, early stopping, or threshold selection.
        m_result = evaluate_frame(model, MESSIDOR, "Messidor-2", tag, seed)
        i_result = evaluate_frame(model, IDRID, "IDRiD", tag, seed)
        fold_results["Messidor-2"].append(m_result)
        fold_results["IDRiD"].append(i_result)
        fold_rows.append({"fold": fold, "seed": seed, "dataset": "Messidor-2", **metrics_from_result(m_result)})
        fold_rows.append({"fold": fold, "seed": seed, "dataset": "IDRiD", **metrics_from_result(i_result)})

        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    fold_df = pd.DataFrame(fold_rows)
    fold_df.to_csv(NEW_TABLE_DIR / "fivefold_foldwise_metrics.csv", index=False)
    key_metrics = ["QWK", "Macro-F1", "Balanced-Acc", "Accuracy", "MAE", "AUROC-OVR", "AUPRC-OVR"]
    summary = _summary_mean_sd(fold_df, "dataset", key_metrics)
    summary.to_csv(NEW_TABLE_DIR / "fivefold_summary_mean_sd.csv", index=False)

    # External predictions can be ensembled because every fold model evaluates the same complete external sets.
    external_ensembles = {}
    for ds in ["Messidor-2", "IDRiD"]:
        ens = ensemble_results(fold_results[ds])
        external_ensembles[ds] = ens
        save_prediction_csv(ens, NEW_PRED_DIR / f"fivefold_models_{_safe_tag(ds)}_ensemble.csv", "five_fold_model_ensemble")
        pd.DataFrame([metrics_from_result(ens)]).to_csv(
            NEW_TABLE_DIR / f"fivefold_models_{_safe_tag(ds)}_ensemble_metrics.csv", index=False
        )

    print("\n5-fold summary:\n", summary.round(4).to_string(index=False))
    return {"fold_metrics": fold_df, "summary": summary, "fold_results": fold_results,
            "external_ensembles": external_ensembles, "checkpoints": records}


FIVEFOLD_STUDY = run_stratified_5fold_external_study()


In [ ]:
# ============================== 25. Validation-fitted temperature scaling ==============================
def _temperature_scale_probs(probs, temperature):
    p = np.asarray(probs, dtype=np.float64)
    p = np.clip(p, 1e-12, 1.0)
    logp = np.log(p)
    z = logp / float(temperature)
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def fit_temperature_from_validation_probs(probs, y):
    """Fit one positive scalar temperature to ensemble log-probabilities on APTOS validation only."""
    p = torch.tensor(np.clip(np.asarray(probs), 1e-12, 1.0), dtype=torch.float64)
    y_t = torch.tensor(np.asarray(y), dtype=torch.long)
    logp = p.log()
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        T = log_t.exp().clamp(0.05, 20.0)
        loss = F.nll_loss(F.log_softmax(logp / T, dim=1), y_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(log_t.detach().exp().clamp(0.05, 20.0).item())


def run_temperature_calibration():
    checkpoints = _full_checkpoint_records()
    val_seed_results = []
    for seed, ckpt in checkpoints:
        model = load_enerdr_checkpoint(ckpt, FULL_SWITCHES)
        val_seed_results.append(predict_enerdr(model, _eval_loader(APTOS_VAL)))
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    val_ens = ensemble_results(val_seed_results)
    temperature = fit_temperature_from_validation_probs(val_ens["probs"], val_ens["y"])

    if FULL_STUDY is None:
        raise RuntimeError("FULL_STUDY is required for canonical ensemble calibration analysis.")
    datasets = {
        "APTOS": FULL_STUDY["internal_ensemble"],
        "Messidor-2": FULL_STUDY["external_ensembles"]["Messidor2_external"],
        "IDRiD": FULL_STUDY["external_ensembles"]["IDRiD_external"],
    }
    rows = []
    calibrated = {}
    for name, result in datasets.items():
        before = metrics_from_result(result)
        after_probs = _temperature_scale_probs(result["probs"], temperature)
        after_result = _result_with_probs(result, after_probs)
        after = metrics_from_result(after_result)
        calibrated[name] = after_result
        for state, m in [("before", before), ("after", after)]:
            rows.append({"dataset": name, "state": state, "temperature": temperature,
                         **{k: m[k] for k in ["ECE", "MCE", "Brier", "NLL", "QWK", "Macro-F1", "AUROC-OVR", "AUPRC-OVR"]}})
        save_prediction_csv(after_result, NEW_PRED_DIR / f"{_safe_tag(name)}_temperature_scaled.csv", "temperature_scaled")

        # Reliability diagram after scaling.
        tab = reliability_bins(after_probs, result["y"], CALIBRATION_BINS)
        tab.to_csv(NEW_TABLE_DIR / f"reliability_bins_{_safe_tag(name)}_after_temperature.csv", index=False)
        valid = tab["count"] > 0
        plt.figure(figsize=(5.5, 5.0))
        plt.plot([0, 1], [0, 1], "--", linewidth=1, label="Perfect calibration")
        plt.plot(tab.loc[valid, "confidence"], tab.loc[valid, "accuracy"], marker="o", label="Temperature scaled")
        plt.xlabel("Mean confidence"); plt.ylabel("Empirical accuracy")
        plt.title(f"Reliability after temperature scaling — {name}")
        plt.xlim(0,1); plt.ylim(0,1); plt.grid(alpha=0.2); plt.legend(); plt.tight_layout()
        plt.savefig(NEW_FIGURE_DIR / f"reliability_temperature_{_safe_tag(name)}.png", dpi=220, bbox_inches="tight")
        plt.close()

    df = pd.DataFrame(rows)
    df.to_csv(NEW_TABLE_DIR / "temperature_scaling_before_after.csv", index=False)
    pd.DataFrame([{"temperature": temperature, "fit_dataset": "APTOS_validation"}]).to_csv(
        NEW_TABLE_DIR / "temperature_scaling_parameter.csv", index=False
    )
    print(f"Validation-fitted temperature: {temperature:.6f}")
    print(df.round(4).to_string(index=False))
    return {"temperature": temperature, "table": df, "calibrated_results": calibrated, "validation_ensemble": val_ens}


CALIBRATION_STUDY = run_temperature_calibration()


In [ ]:
# ============================== 26. XAI: Grad-CAM, particle attention, combined evidence, occlusion and lesion localization ==============================
def _normalize_heatmap(x):
    x = np.asarray(x, dtype=np.float32)
    x = x - np.min(x)
    den = float(np.max(x))
    return x / den if den > 1e-12 else np.zeros_like(x)


def _resize_heatmap(hm, size=IMAGE_SIZE):
    im = Image.fromarray(np.uint8(np.clip(hm, 0, 1) * 255))
    im = im.resize((size, size), Image.Resampling.BILINEAR)
    return np.asarray(im, dtype=np.float32) / 255.0


def _denormalize_tensor(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(3,1,1)
    std = torch.tensor(IMAGENET_STD, device=x.device).view(3,1,1)
    return (x * std + mean).clamp(0,1)


def gradcam_and_particle_attention(model, x, class_idx=None):
    model.eval(); model.zero_grad(set_to_none=True); x = x.to(DEVICE)
    with torch.enable_grad():
        out = model(x)
        feat = out["feature_map"]; feat.retain_grad()
        if class_idx is None:
            class_idx = int(out["probs"].argmax(1)[0].item())
        out["logits"][0, class_idx].backward(retain_graph=True)
        grad = feat.grad[0]; fmap = feat.detach()[0]
        weights = grad.mean(dim=(1,2), keepdim=True)
        cam = torch.relu((weights * fmap).sum(0)).detach().cpu().numpy()
    cam = _resize_heatmap(_normalize_heatmap(cam), IMAGE_SIZE)
    with torch.no_grad():
        out2 = model(x)
        maps = out2["particle_maps"][0]
        weights = out2["particle_burden"][0].detach().clamp_min(0)
        if float(weights.sum()) <= 1e-12:
            weights = torch.ones_like(weights)
        weights = weights / weights.sum()
        att = (maps * weights[:, None, None]).sum(0).detach().cpu().numpy()
        att = _resize_heatmap(_normalize_heatmap(att), IMAGE_SIZE)
        base_prob = float(out2["probs"][0, class_idx].item())
    combined = _normalize_heatmap(0.5 * cam + 0.5 * att)
    return cam, att, combined, class_idx, base_prob


def _occlusion_drop(model, x, heatmap, class_idx, top_fraction=XAI_TOP_FRACTION, seed=0):
    flat = heatmap.reshape(-1)
    k = max(1, int(round(len(flat) * float(top_fraction))))
    top_idx = np.argpartition(flat, -k)[-k:]
    salient_mask = np.zeros(len(flat), dtype=bool); salient_mask[top_idx] = True
    salient_mask = salient_mask.reshape(IMAGE_SIZE, IMAGE_SIZE)
    rng = np.random.default_rng(seed)
    random_idx = rng.choice(len(flat), size=k, replace=False)
    random_mask = np.zeros(len(flat), dtype=bool); random_mask[random_idx] = True
    random_mask = random_mask.reshape(IMAGE_SIZE, IMAGE_SIZE)
    with torch.no_grad():
        base = float(model(x.to(DEVICE))["probs"][0, class_idx].item())
        xs = x.clone().to(DEVICE); xr = x.clone().to(DEVICE)
        sm = torch.from_numpy(salient_mask).to(DEVICE); rm = torch.from_numpy(random_mask).to(DEVICE)
        xs[0, :, sm] = 0.0; xr[0, :, rm] = 0.0
        ps = float(model(xs)["probs"][0, class_idx].item())
        pr = float(model(xr)["probs"][0, class_idx].item())
    return base - ps, base - pr, salient_mask


def _idrid_mask_candidates():
    roots = []
    p = IDRID_ROOT.resolve()
    roots.extend([p] + list(p.parents[:4]))
    lesion_words = ("microaneur", "haemorr", "hemorr", "exud", "mask", "segmentation", "lesion")
    candidates, seen = [], set()
    for root in roots:
        if not root.exists():
            continue
        for q in root.rglob("*"):
            if q.is_file() and q.suffix.lower() in IMAGE_SUFFIXES:
                low = str(q).lower()
                if any(w in low for w in lesion_words) and q not in seen:
                    seen.add(q); candidates.append(q)
    return candidates


def _normalize_stem(s):
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())


def build_idrid_union_mask_index():
    candidates = _idrid_mask_candidates()
    if not candidates:
        raise RuntimeError("No IDRiD lesion-annotation files were found. XAI localization requires the real lesion masks.")
    cand_norm = [(_normalize_stem(p.stem), p) for p in candidates]
    idx = {}
    for image_id in IDRID.image_id.astype(str):
        key = _normalize_stem(Path(image_id).stem)
        matches = [p for stem, p in cand_norm if key and key in stem]
        if matches:
            idx[str(image_id)] = matches
    if not idx:
        raise RuntimeError("IDRiD lesion files were found, but none could be matched to grading image IDs.")
    print(f"Lesion-mask mapping found for {len(idx)} / {len(IDRID)} IDRiD images.")
    return idx


def load_union_lesion_mask(paths):
    union = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    for p in paths:
        with Image.open(p) as im:
            arr = np.asarray(im.convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST))
        union |= arr > 0
    if union.sum() == 0:
        raise RuntimeError(f"Matched lesion masks are empty: {paths}")
    return union


def run_xai_localization():
    checkpoints = _full_checkpoint_records()
    best = max((float(torch.load(p, map_location="cpu").get("val_qwk", -np.inf)), int(seed), p)
               for seed, p in checkpoints)
    _, best_seed, best_ckpt = best
    model = load_enerdr_checkpoint(best_ckpt, FULL_SWITCHES)
    mask_index = build_idrid_union_mask_index()

    eligible = IDRID[IDRID.image_id.astype(str).isin(mask_index.keys())].copy()
    if len(eligible) < NUM_CLASSES:
        raise RuntimeError("Too few lesion-annotated IDRiD grading images for stratified XAI analysis.")
    parts = []
    per_grade = max(1, XAI_MAX_SAMPLES // NUM_CLASSES)
    for g, grp in eligible.groupby("grade"):
        parts.append(grp.sample(n=min(per_grade, len(grp)), random_state=SPLIT_SEED + int(g)))
    sample_df = pd.concat(parts, ignore_index=True).head(XAI_MAX_SAMPLES)

    rows = []; example_count = 0
    for j, row in sample_df.iterrows():
        ds = ExperimentRetinaDataset(pd.DataFrame([row]), train=False, path_col="cached_path")
        x, y, image_id = ds[0]; x = x.unsqueeze(0)
        cam, att, combined, pred_class, base_prob = gradcam_and_particle_attention(model, x)
        lesion = load_union_lesion_mask(mask_index[str(image_id)])

        def loc_metrics(hm, top_mask):
            peak = np.unravel_index(np.argmax(hm), hm.shape)
            pointing = float(bool(lesion[peak]))
            overlap = float((top_mask & lesion).sum() / max(top_mask.sum(), 1))
            return pointing, overlap

        for method, hm, seed_offset in [
            ("Grad-CAM", cam, 0),
            ("EnerDR_particle_attention", att, 10000),
            ("Combined_GradCAM_particle_attention", combined, 20000),
        ]:
            drop, rand_drop, top = _occlusion_drop(model, x, hm, pred_class, seed=SPLIT_SEED + seed_offset + j)
            point, overlap = loc_metrics(hm, top)
            rows.append({"image_id": image_id, "grade": int(y), "predicted_grade": pred_class, "method": method,
                         "base_probability": base_prob, "salient_occlusion_drop": drop,
                         "random_occlusion_drop": rand_drop, "pointing_game_hit": point,
                         "top20_lesion_overlap": overlap})

        if example_count < 15:
            rgb = _denormalize_tensor(x[0].to(DEVICE)).permute(1,2,0).detach().cpu().numpy()
            fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
            axes[0].imshow(rgb); axes[0].set_title(f"Image\ntrue={y}, pred={pred_class}")
            for ax, hm, title in [(axes[1],cam,"Grad-CAM"),(axes[2],att,"Particle attention"),(axes[3],combined,"Combined")]:
                ax.imshow(rgb); ax.imshow(hm, alpha=0.45, cmap="jet"); ax.set_title(title)
            axes[4].imshow(rgb); axes[4].imshow(lesion, alpha=0.45, cmap="Reds"); axes[4].set_title("Lesion-mask union")
            for ax in axes: ax.axis("off")
            plt.tight_layout(); plt.savefig(NEW_FIGURE_DIR / f"xai_{example_count:02d}_{_safe_tag(image_id)}.png", dpi=220, bbox_inches="tight"); plt.close(fig)
            example_count += 1

    df = pd.DataFrame(rows)
    numeric = df.select_dtypes(include=[np.number])
    if numeric.isna().any().any() or not np.isfinite(numeric.to_numpy()).all():
        raise RuntimeError("XAI table contains missing or non-finite measurements.")
    df.to_csv(NEW_TABLE_DIR / "xai_localization_per_image.csv", index=False)
    summary = df.groupby("method", as_index=False).agg(
        n=("image_id", "count"),
        salient_occlusion_drop_mean=("salient_occlusion_drop", "mean"),
        random_occlusion_drop_mean=("random_occlusion_drop", "mean"),
        pointing_game_hit_rate=("pointing_game_hit", "mean"),
        top20_lesion_overlap_mean=("top20_lesion_overlap", "mean"),
    )
    summary["occlusion_faithfulness_ratio"] = summary["salient_occlusion_drop_mean"] / summary["random_occlusion_drop_mean"].clip(lower=1e-12)
    summary.to_csv(NEW_TABLE_DIR / "xai_localization_summary.csv", index=False)
    print(summary.round(4).to_string(index=False))
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return {"per_image": df, "summary": summary, "checkpoint_seed": best_seed, "mask_count": len(mask_index)}


XAI_STUDY = run_xai_localization()


In [ ]:
# ============================== 27. Cross-dataset domain-shift / applicability analysis ==============================
@torch.no_grad()
def extract_encoder_embeddings(model, frame, path_col="cached_path"):
    loader = DataLoader(
        ExperimentRetinaDataset(frame, train=False, path_col=path_col),
        batch_size=DOMAIN_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    ids, ys, embs = [], [], []
    model.eval()
    for x, y, image_ids in loader:
        x = x.to(DEVICE, non_blocking=True)
        fmap = model.encoder(x)
        e = F.adaptive_avg_pool2d(fmap, 1).flatten(1)
        e = F.normalize(e, dim=1)
        embs.append(e.cpu().numpy())
        ys.extend(y.numpy().tolist())
        ids.extend(list(image_ids))
    return {"ids": ids, "y": np.asarray(ys, dtype=int), "emb": np.concatenate(embs, axis=0)}


def average_embeddings_across_checkpoints(frame, checkpoint_records, path_col="cached_path"):
    avg = None; ids_ref = None; y_ref = None
    for seed, ckpt in checkpoint_records:
        model = load_enerdr_checkpoint(ckpt, FULL_SWITCHES)
        r = extract_encoder_embeddings(model, frame, path_col=path_col)
        if avg is None:
            ids_ref, y_ref = list(r["ids"]), r["y"]
            avg = r["emb"].astype(np.float64)
        else:
            assert ids_ref == list(r["ids"])
            assert np.array_equal(y_ref, r["y"])
            avg += r["emb"]
        del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    avg /= len(checkpoint_records)
    avg /= np.maximum(np.linalg.norm(avg, axis=1, keepdims=True), 1e-12)
    return {"ids": ids_ref, "y": y_ref, "emb": avg.astype(np.float32)}


def max_cosine_to_train(query_emb, train_emb, chunk=512):
    vals = []
    train_t = torch.tensor(train_emb, dtype=torch.float32, device=DEVICE)
    train_t = F.normalize(train_t, dim=1)
    for i in range(0, len(query_emb), chunk):
        q = torch.tensor(query_emb[i:i+chunk], dtype=torch.float32, device=DEVICE)
        q = F.normalize(q, dim=1)
        vals.append((q @ train_t.T).max(dim=1).values.cpu().numpy())
    return np.concatenate(vals)


def _align_result_by_ids(result, ids):
    pos = {str(k): i for i, k in enumerate(result["ids"])}
    take = np.asarray([pos[str(k)] for k in ids], dtype=int)
    return {
        "ids": [result["ids"][i] for i in take],
        "y": np.asarray(result["y"])[take],
        "pred": np.asarray(result["pred"])[take],
        "probs": np.asarray(result["probs"])[take],
        "burden": np.asarray(result["burden"])[take] if result.get("burden") is not None else None,
    }


def run_domain_shift_analysis():
    if FULL_STUDY is None:
        raise RuntimeError("FULL_STUDY required for domain-shift analysis.")
    checkpoints = _full_checkpoint_records()
    train_e = average_embeddings_across_checkpoints(APTOS_TRAIN, checkpoints)
    datasets = {
        "APTOS": (APTOS_TEST, FULL_STUDY["internal_ensemble"]),
        "Messidor-2": (MESSIDOR, FULL_STUDY["external_ensembles"]["Messidor2_external"]),
        "IDRiD": (IDRID, FULL_STUDY["external_ensembles"]["IDRiD_external"]),
    }
    rows = []; subset_rows = []
    for name, (frame, result) in datasets.items():
        e = average_embeddings_across_checkpoints(frame, checkpoints)
        sim = max_cosine_to_train(e["emb"], train_e["emb"])
        dist = 1.0 - sim
        aligned = _align_result_by_ids(result, e["ids"])
        err = np.abs(np.asarray(aligned["pred"]) - np.asarray(aligned["y"]))
        conf = np.asarray(aligned["probs"]).max(1)
        for i, image_id in enumerate(e["ids"]):
            rows.append({"dataset": name, "image_id": image_id, "grade": int(e["y"][i]),
                         "max_cosine_similarity": float(sim[i]), "distance": float(dist[i]),
                         "absolute_ordinal_error": int(err[i]), "confidence": float(conf[i])})

        median_sim = float(np.median(sim))
        for label, mask in [("near_domain", sim >= median_sim), ("far_domain", sim < median_sim)]:
            idx = np.where(mask)[0]
            sub = {
                "y": np.asarray(aligned["y"])[idx], "pred": np.asarray(aligned["pred"])[idx],
                "probs": np.asarray(aligned["probs"])[idx],
                "burden": np.asarray(aligned["burden"])[idx] if aligned.get("burden") is not None else None,
            }
            m = metrics_from_result(sub)
            subset_rows.append({"dataset": name, "domain_subset": label, "n": len(idx),
                                "mean_similarity": float(sim[idx].mean()), "median_split_similarity": median_sim,
                                **{k: m[k] for k in ["QWK", "Macro-F1", "Balanced-Acc", "Accuracy", "MAE"]}})

    per_image = pd.DataFrame(rows)
    subsets = pd.DataFrame(subset_rows)
    per_image.to_csv(NEW_TABLE_DIR / "domain_shift_per_image.csv", index=False)
    subsets.to_csv(NEW_TABLE_DIR / "domain_shift_near_far_performance.csv", index=False)
    corr = pd.DataFrame([{
        "distance_vs_absolute_error_spearman": per_image["distance"].corr(per_image["absolute_ordinal_error"], method="spearman"),
        "distance_vs_confidence_spearman": per_image["distance"].corr(per_image["confidence"], method="spearman"),
    }])
    corr.to_csv(NEW_TABLE_DIR / "domain_shift_correlations.csv", index=False)
    summary = per_image.groupby("dataset", as_index=False).agg(
        n=("image_id", "count"), mean_max_cosine_similarity=("max_cosine_similarity", "mean"),
        median_max_cosine_similarity=("max_cosine_similarity", "median"), mean_distance=("distance", "mean"),
    )
    summary.to_csv(NEW_TABLE_DIR / "domain_shift_summary.csv", index=False)

    plt.figure(figsize=(6.4, 4.2))
    for name, g in per_image.groupby("dataset"):
        plt.hist(g["max_cosine_similarity"], bins=30, alpha=0.35, density=True, label=name)
    plt.xlabel("Maximum cosine similarity to APTOS train")
    plt.ylabel("Density"); plt.title("Feature-space domain similarity"); plt.legend(); plt.tight_layout()
    plt.savefig(NEW_FIGURE_DIR / "domain_shift_similarity_distributions.png", dpi=220, bbox_inches="tight")
    plt.close()
    print(summary.round(4).to_string(index=False))
    print(subsets.round(4).to_string(index=False))
    return {"per_image": per_image, "summary": summary, "subsets": subsets, "correlations": corr}


DOMAIN_SHIFT_STUDY = run_domain_shift_analysis()


In [ ]:
# ============================== 28. Preprocessing ablation ==============================
def _retinal_support_crop_pil(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        arr = np.asarray(im)
    gray = arr.mean(axis=2)
    threshold = max(8.0, float(np.percentile(gray, 10)))
    mask = gray > threshold
    if mask.mean() >= 0.05:
        ys, xs = np.where(mask)
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        h, w = arr.shape[:2]
        my, mx = max(2, int(0.01*h)), max(2, int(0.01*w))
        y0, y1 = max(0, y0-my), min(h, y1+my)
        x0, x1 = max(0, x0-mx), min(w, x1+mx)
        return Image.fromarray(arr[y0:y1, x0:x1])
    return Image.fromarray(arr)


def preprocess_ablation_image(path, variant):
    if variant == "direct_resize":
        with Image.open(path) as im:
            return im.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    crop = _retinal_support_crop_pil(path)
    if variant == "support_crop_resize":
        return crop.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
    if variant == "crop_squarepad_bilinear":
        w, h = crop.size; side = max(w, h)
        canvas = Image.new("RGB", (side, side), (0,0,0))
        canvas.paste(crop, ((side-w)//2, (side-h)//2))
        return canvas.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    if variant == "full_rspin_lanczos":
        return retinal_support_preprocess(path, IMAGE_SIZE)
    raise ValueError(variant)


def cache_preprocess_variant(frame, dataset_name, variant):
    if variant == "full_rspin_lanczos":
        out = frame.copy(); out["experiment_path"] = out["cached_path"]
        return out
    target = NEW_CACHE_DIR / "preprocessing" / variant / _safe_tag(dataset_name)
    target.mkdir(parents=True, exist_ok=True)
    out = frame.copy(); paths = []
    for path in out.path.astype(str):
        p = Path(path); st = p.stat()
        token = f"{p.resolve()}|{st.st_size}|{st.st_mtime_ns}|{variant}|{IMAGE_SIZE}"
        dst = target / f"{hashlib.sha1(token.encode()).hexdigest()}.png"
        if not dst.exists():
            preprocess_ablation_image(path, variant).save(dst, format="PNG", optimize=True)
        paths.append(str(dst))
    out["experiment_path"] = paths
    return out


def _run_multiseed_variant(tag, train_df, val_df, test_df, external_frames,
                           switches=FULL_SWITCHES, path_col="cached_path", aug_mode="geom_photo"):
    seed_results = []; ext_seed_results = {k: [] for k in external_frames}; rows=[]; ckpts=[]
    for seed in NEW_EXPERIMENT_SEEDS:
        model, ckpt, test_result, info = fit_enerdr_frames(
            train_df, val_df, test_df, seed, tag=tag, switches=switches,
            path_col=path_col, aug_mode=aug_mode,
        )
        ckpts.append((seed, ckpt)); seed_results.append(test_result)
        rows.append({"variant": tag, "seed": seed, "dataset": "APTOS", **metrics_from_result(test_result), **info})
        for ds, frame in external_frames.items():
            r = evaluate_frame(model, frame, ds, tag, seed, path_col=path_col)
            ext_seed_results[ds].append(r)
            rows.append({"variant": tag, "seed": seed, "dataset": ds, **metrics_from_result(r)})
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"rows": pd.DataFrame(rows), "internal_seed_results": seed_results,
            "internal_ensemble": ensemble_results(seed_results),
            "external_seed_results": ext_seed_results,
            "external_ensembles": {k: ensemble_results(v) for k,v in ext_seed_results.items()}, "checkpoints": ckpts}


def run_preprocessing_ablation():
    variants = ["direct_resize", "support_crop_resize", "crop_squarepad_bilinear", "full_rspin_lanczos"]
    all_rows = []; ensemble_rows = []
    for variant in variants:
        print(f"\n===== Preprocessing ablation: {variant} =====")
        if variant == "full_rspin_lanczos" and FULL_STUDY is not None:
            # Reuse canonical full-model study because this is exactly the canonical preprocessing/training protocol.
            seed_df = FULL_STUDY["metrics"].copy()
            map_split = {"APTOS_internal_test":"APTOS", "Messidor2_external":"Messidor-2", "IDRiD_external":"IDRiD"}
            for _, r in seed_df.iterrows():
                if r["split"] in map_split:
                    all_rows.append({"variant": variant, "seed": int(r["seed"]), "dataset": map_split[r["split"]], **{k:r[k] for k in r.index if k not in {"seed","split"}}})
            ensembles = {"APTOS": FULL_STUDY["internal_ensemble"],
                         "Messidor-2": FULL_STUDY["external_ensembles"]["Messidor2_external"],
                         "IDRiD": FULL_STUDY["external_ensembles"]["IDRiD_external"]}
        else:
            frames = {
                "APTOS_TRAIN": cache_preprocess_variant(APTOS_TRAIN, "APTOS_TRAIN", variant),
                "APTOS_VAL": cache_preprocess_variant(APTOS_VAL, "APTOS_VAL", variant),
                "APTOS_TEST": cache_preprocess_variant(APTOS_TEST, "APTOS_TEST", variant),
                "Messidor-2": cache_preprocess_variant(MESSIDOR, "Messidor-2", variant),
                "IDRiD": cache_preprocess_variant(IDRID, "IDRiD", variant),
            }
            study = _run_multiseed_variant(
                f"Preproc_{variant}", frames["APTOS_TRAIN"], frames["APTOS_VAL"], frames["APTOS_TEST"],
                {"Messidor-2":frames["Messidor-2"], "IDRiD":frames["IDRiD"]},
                switches=FULL_SWITCHES, path_col="experiment_path", aug_mode="geom_photo",
            )
            all_rows.extend(study["rows"].to_dict("records"))
            ensembles = {"APTOS":study["internal_ensemble"], **study["external_ensembles"]}
        for ds, result in ensembles.items():
            m = metrics_from_result(result)
            ensemble_rows.append({"variant":variant, "dataset":ds, **{k:m[k] for k in ["QWK","Macro-F1","Balanced-Acc","Accuracy","MAE","AUROC-OVR"]}})

    raw = pd.DataFrame(all_rows); ens = pd.DataFrame(ensemble_rows)
    raw.to_csv(NEW_TABLE_DIR / "preprocessing_ablation_seed_metrics.csv", index=False)
    ens.to_csv(NEW_TABLE_DIR / "preprocessing_ablation_ensemble_metrics.csv", index=False)
    print(ens.round(4).to_string(index=False))
    return {"seed_metrics":raw, "ensemble_metrics":ens}


PREPROCESSING_STUDY = run_preprocessing_ablation()


In [ ]:
# ============================== 29. Augmentation ablation ==============================
def _checkpoint_results_for_tag(tag, switches, external=True):
    internal=[]; external_results={"Messidor-2":[], "IDRiD":[]}; rows=[]
    for seed in NEW_EXPERIMENT_SEEDS:
        p = CHECKPOINT_DIR / f"EnerDR_{tag}_seed{seed}.pt"
        if not p.exists():
            return None
        model = load_enerdr_checkpoint(p, switches)
        apt = predict_enerdr(model, _eval_loader(APTOS_TEST))
        internal.append(apt); rows.append({"variant":tag,"seed":seed,"dataset":"APTOS",**metrics_from_result(apt)})
        if external:
            for ds, frame in [("Messidor-2", MESSIDOR), ("IDRiD", IDRID)]:
                rr = predict_enerdr(model, _eval_loader(frame)); external_results[ds].append(rr)
                rows.append({"variant":tag,"seed":seed,"dataset":ds,**metrics_from_result(rr)})
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"rows":pd.DataFrame(rows), "internal_ensemble":ensemble_results(internal),
            "external_ensembles":{k:ensemble_results(v) for k,v in external_results.items()}}


def run_augmentation_ablation():
    configurations = [
        ("NoAug_NoPSIC", "none", ModelSwitches(use_psic=False)),
        ("GeomOnly_NoPSIC", "geom", ModelSwitches(use_psic=False)),
        ("GeomPhoto_NoPSIC", "geom_photo", ModelSwitches(use_psic=False)),
        ("FullAug_PSIC", "geom_photo", FULL_SWITCHES),
    ]
    seed_rows=[]; ensemble_rows=[]
    for name, aug_mode, switches in configurations:
        print(f"\n===== Augmentation ablation: {name} =====")
        study=None
        if name == "FullAug_PSIC" and FULL_STUDY is not None:
            rows=[]
            for _, r in FULL_STUDY["metrics"].iterrows():
                ds={"APTOS_internal_test":"APTOS","Messidor2_external":"Messidor-2","IDRiD_external":"IDRiD"}.get(r["split"])
                if ds: rows.append({"variant":name,"seed":int(r["seed"]),"dataset":ds,**{k:r[k] for k in r.index if k not in {"seed","split"}}})
            study={"rows":pd.DataFrame(rows), "internal_ensemble":FULL_STUDY["internal_ensemble"],
                   "external_ensembles":{"Messidor-2":FULL_STUDY["external_ensembles"]["Messidor2_external"],
                                         "IDRiD":FULL_STUDY["external_ensembles"]["IDRiD_external"]}}
        elif name == "GeomPhoto_NoPSIC":
            reused=_checkpoint_results_for_tag("No_PSIC", switches)
            if reused is not None: study=reused
        if study is None:
            study=_run_multiseed_variant(
                f"Aug_{name}", APTOS_TRAIN, APTOS_VAL, APTOS_TEST,
                {"Messidor-2":MESSIDOR,"IDRiD":IDRID}, switches=switches,
                path_col="cached_path", aug_mode=aug_mode,
            )
        seed_rows.extend(study["rows"].to_dict("records"))
        ensembles={"APTOS":study["internal_ensemble"], **study["external_ensembles"]}
        for ds,res in ensembles.items():
            m=metrics_from_result(res)
            ensemble_rows.append({"variant":name,"dataset":ds,**{k:m[k] for k in ["QWK","Macro-F1","Balanced-Acc","Accuracy","MAE","AUROC-OVR"]}})

    seed_df=pd.DataFrame(seed_rows); ens_df=pd.DataFrame(ensemble_rows)
    seed_df.to_csv(NEW_TABLE_DIR / "augmentation_ablation_seed_metrics.csv",index=False)
    ens_df.to_csv(NEW_TABLE_DIR / "augmentation_ablation_ensemble_metrics.csv",index=False)
    print(ens_df.round(4).to_string(index=False))
    return {"seed_metrics":seed_df,"ensemble_metrics":ens_df}


AUGMENTATION_STUDY = run_augmentation_ablation()


In [ ]:
# ============================== 30. RDR/VTDR operating-point analysis ==============================
def _binary_metrics_from_threshold(y_binary, positive_prob, threshold):
    yb=np.asarray(y_binary,dtype=bool); p=np.asarray(positive_prob,float)
    yp=p>=float(threshold)
    tp=int(np.sum(yb & yp)); tn=int(np.sum(~yb & ~yp)); fp=int(np.sum(~yb & yp)); fn=int(np.sum(yb & ~yp))
    sens=tp/max(tp+fn,1); spec=tn/max(tn+fp,1); ppv=tp/max(tp+fp,1); npv=tn/max(tn+fn,1)
    return {"TP":tp,"TN":tn,"FP":fp,"FN":fn,"Sensitivity":sens,"Specificity":spec,
            "PPV":ppv,"NPV":npv,"Accuracy":(tp+tn)/max(len(yb),1),"Balanced_Accuracy":0.5*(sens+spec)}


def select_operating_threshold(y_binary, positive_prob, mode="balanced", sensitivity_target=None):
    yb=np.asarray(y_binary,dtype=bool); p=np.asarray(positive_prob,float)
    # Candidate thresholds include boundaries around every distinct probability; selection uses validation only.
    vals=np.unique(np.clip(p,0,1))
    candidates=np.unique(np.r_[0.0, vals, 1.0])
    scored=[]
    for t in candidates:
        m=_binary_metrics_from_threshold(yb,p,t); scored.append((float(t),m))
    if mode=="balanced":
        return max(scored,key=lambda z:(z[1]["Balanced_Accuracy"],z[1]["Sensitivity"],z[1]["Specificity"]))[0]
    if mode=="high_sensitivity":
        eligible=[z for z in scored if z[1]["Sensitivity"] >= float(sensitivity_target)]
        if eligible:
            return max(eligible,key=lambda z:(z[1]["Specificity"],z[1]["Balanced_Accuracy"]))[0]
        return max(scored,key=lambda z:(z[1]["Sensitivity"],z[1]["Specificity"]))[0]
    raise ValueError(mode)


def _endpoint_probability(result, grade_threshold):
    return np.asarray(result["probs"])[:, grade_threshold:].sum(1)


def run_operating_point_analysis():
    checkpoints=_full_checkpoint_records(); val_results=[]
    for seed,ckpt in checkpoints:
        model=load_enerdr_checkpoint(ckpt,FULL_SWITCHES)
        val_results.append(predict_enerdr(model,_eval_loader(APTOS_VAL)))
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    val_ens=ensemble_results(val_results)

    if FULL_STUDY is None: raise RuntimeError("FULL_STUDY required.")
    eval_sets={"APTOS":FULL_STUDY["internal_ensemble"],
               "Messidor-2":FULL_STUDY["external_ensembles"]["Messidor2_external"],
               "IDRiD":FULL_STUDY["external_ensembles"]["IDRiD_external"]}
    endpoint_specs=[("RDR",2,RDR_HIGH_SENSITIVITY_TARGET),("VTDR",3,VTDR_HIGH_SENSITIVITY_TARGET)]
    rows=[]; threshold_rows=[]
    for endpoint,grade_thr,sens_target in endpoint_specs:
        yv=np.asarray(val_ens["y"])>=grade_thr; pv=_endpoint_probability(val_ens,grade_thr)
        thresholds={
            "balanced":select_operating_threshold(yv,pv,"balanced"),
            "high_sensitivity":select_operating_threshold(yv,pv,"high_sensitivity",sens_target),
        }
        for name,t in thresholds.items():
            threshold_rows.append({"endpoint":endpoint,"operating_point":name,"threshold":t,
                                   "selected_on":"APTOS_validation", "sensitivity_target":sens_target if name=="high_sensitivity" else "not_applicable"})
            for ds,result in eval_sets.items():
                y=np.asarray(result["y"])>=grade_thr; p=_endpoint_probability(result,grade_thr)
                m=_binary_metrics_from_threshold(y,p,t)
                if len(np.unique(y.astype(int))) < 2:
                    raise RuntimeError(f"{ds} {endpoint}: endpoint has only one class, AUROC/AUPRC are undefined.")
                auroc=float(roc_auc_score(y.astype(int),p))
                auprc=float(average_precision_score(y.astype(int),p))
                rows.append({"dataset":ds,"endpoint":endpoint,"operating_point":name,"threshold":t,
                             **m,"AUROC":auroc,"AUPRC":auprc})
    df=pd.DataFrame(rows); th=pd.DataFrame(threshold_rows)
    df.to_csv(NEW_TABLE_DIR / "clinical_operating_points_RDR_VTDR.csv",index=False)
    th.to_csv(NEW_TABLE_DIR / "clinical_operating_point_thresholds.csv",index=False)
    print(th.round(5).to_string(index=False)); print(df.round(4).to_string(index=False))
    return {"thresholds":th,"results":df,"validation_ensemble":val_ens}


OPERATING_POINT_STUDY = run_operating_point_analysis()


In [ ]:
# ============================== 31. Extended-experiment manifest and output index ==============================
def write_new_experiment_manifest():
    manifest = {
        "generated_at_unix": time.time(),
        "method": "EnerDR-Net",
        "results_computed_from_run_artifacts": True,
        "development_dataset": "APTOS",
        "external_datasets": ["Messidor-2", "IDRiD"],
        "cv_policy": "5-fold stratified APTOS outer CV; inner APTOS validation only; external sets frozen",
        "temperature_policy": "single scalar temperature fitted on APTOS validation predictions only",
        "operating_point_policy": "RDR/VTDR thresholds selected on APTOS validation only and frozen externally",
        "preprocessing_variants": ["direct_resize", "support_crop_resize", "crop_squarepad_bilinear", "full_rspin_lanczos"],
        "augmentation_variants": ["NoAug_NoPSIC", "GeomOnly_NoPSIC", "GeomPhoto_NoPSIC", "FullAug_PSIC"],
        "xai_policy": "Grad-CAM + intrinsic particle attention + salient-vs-random occlusion; lesion metrics only when masks are discoverable",
        "output_directory": str(NEW_TABLE_DIR),
        "flags": {
            "RUN_STRATIFIED_5FOLD_CV": RUN_STRATIFIED_5FOLD_CV,
            "RUN_TEMPERATURE_CALIBRATION": RUN_TEMPERATURE_CALIBRATION,
            "RUN_XAI_LOCALIZATION": RUN_XAI_LOCALIZATION,
            "RUN_DOMAIN_SHIFT": RUN_DOMAIN_SHIFT,
            "RUN_PREPROCESSING_ABLATION": RUN_PREPROCESSING_ABLATION,
            "RUN_AUGMENTATION_ABLATION": RUN_AUGMENTATION_ABLATION,
            "RUN_OPERATING_POINTS": RUN_OPERATING_POINTS,
        },
    }
    (NEW_TABLE_DIR / "new_experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    files = sorted([p.name for p in NEW_TABLE_DIR.glob("*") if p.is_file()])
    pd.DataFrame({"paper_ready_output": files}).to_csv(NEW_TABLE_DIR / "paper_ready_output_index.csv", index=False)
    print("\nNew paper-experiment tables written to:", NEW_TABLE_DIR)
    print("Figures written to:", NEW_FIGURE_DIR)
    return manifest


NEW_EXPERIMENT_MANIFEST = write_new_experiment_manifest()


### New paper-ready outputs

The additional experiments write their numerical results to `tables/extended_experiments/`, figures to `figures/extended_experiments/`, predictions to `predictions/extended_experiments/`, and dedicated checkpoints to `checkpoints/extended_experiments/`.

Important: the notebook contains **no target manuscript values** for these new experiments. The paper tables must be populated from the CSV files generated by the actual run.


## Final completeness audit
This final block verifies that the requested experimental outputs were actually generated from the run, that required files are present, and that result tables do not contain missing or non-finite numeric values.


In [ ]:
# ============================== 32. Final completeness audit and result catalog ==============================
def _require_file(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Required result file is missing or empty: {path}")
    return path


def _audit_csv(path, allow_text_na=False):
    path = _require_file(path)
    df = pd.read_csv(path)
    if len(df) == 0:
        raise RuntimeError(f"Result table has no rows: {path}")
    numeric = df.select_dtypes(include=[np.number])
    if numeric.isna().any().any():
        bad = numeric.columns[numeric.isna().any()].tolist()
        raise RuntimeError(f"Numeric missing values in {path.name}: {bad}")
    if numeric.size and not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise RuntimeError(f"Non-finite numeric values in {path.name}")
    if not allow_text_na:
        text_cols = [c for c in df.columns if c not in numeric.columns]
        for c in text_cols:
            if df[c].isna().any():
                raise RuntimeError(f"Missing text values in {path.name}, column {c}")
    return df


def run_complete_output_audit():
    required_tables = [
        TABLE_DIR / "dataset_summary.csv",
        TABLE_DIR / "mathematical_implementation_integrity.csv",
        TABLE_DIR / "gradient_coverage.csv",
        TABLE_DIR / "EnerDR_Full_five_seed_metrics.csv",
        TABLE_DIR / "EnerDR_Full_five_seed_summary.csv",
        TABLE_DIR / "ablation_multiseed_metrics.csv",
        TABLE_DIR / "ablation_multiseed_summary.csv",
        TABLE_DIR / "baseline_five_seed_metrics.csv",
        TABLE_DIR / "model_parameter_counts.csv",
        TABLE_DIR / "paired_bootstrap_Full_vs_all.csv",
        TABLE_DIR / "paper_master_performance.csv",
        TABLE_DIR / "paper_five_seed_QWK.csv",
        TABLE_DIR / "paper_per_class_metrics.csv",
        TABLE_DIR / "paper_clinical_RDR_VTDR.csv",
        TABLE_DIR / "paper_ordinal_error_distribution.csv",
        TABLE_DIR / "paper_failure_case_analysis.csv",
        TABLE_DIR / "paper_selective_prediction.csv",
        TABLE_DIR / "paper_disease_burden_progression.csv",
        TABLE_DIR / "paper_ablation_summary.csv",
        TABLE_DIR / "paper_baseline_generalization.csv",
        TABLE_DIR / "paper_LREF_energy_dynamics.csv",
        TABLE_DIR / "RDEM_structural_diagnostics_summary.csv",
        TABLE_DIR / "RDEM_interaction_burden_by_grade.csv",
        TABLE_DIR / "DEGD_burden_center_alignment.csv",
        TABLE_DIR / "paper_cross_dataset_generalization.csv",
        TABLE_DIR / "paper_training_time_per_epoch.csv",
        TABLE_DIR / "paper_training_convergence_summary.csv",
        TABLE_DIR / "paper_hyperparameter_sensitivity.csv",
        TABLE_DIR / "paper_efficiency_profile.csv",
        NEW_TABLE_DIR / "fivefold_foldwise_metrics.csv",
        NEW_TABLE_DIR / "fivefold_summary_mean_sd.csv",
        NEW_TABLE_DIR / "temperature_scaling_before_after.csv",
        NEW_TABLE_DIR / "temperature_scaling_parameter.csv",
        NEW_TABLE_DIR / "xai_localization_per_image.csv",
        NEW_TABLE_DIR / "xai_localization_summary.csv",
        NEW_TABLE_DIR / "domain_shift_per_image.csv",
        NEW_TABLE_DIR / "domain_shift_near_far_performance.csv",
        NEW_TABLE_DIR / "domain_shift_correlations.csv",
        NEW_TABLE_DIR / "domain_shift_summary.csv",
        NEW_TABLE_DIR / "preprocessing_ablation_seed_metrics.csv",
        NEW_TABLE_DIR / "preprocessing_ablation_ensemble_metrics.csv",
        NEW_TABLE_DIR / "augmentation_ablation_seed_metrics.csv",
        NEW_TABLE_DIR / "augmentation_ablation_ensemble_metrics.csv",
        NEW_TABLE_DIR / "clinical_operating_points_RDR_VTDR.csv",
        NEW_TABLE_DIR / "clinical_operating_point_thresholds.csv",
    ]
    audited = []
    for p in required_tables:
        # Threshold metadata intentionally contains a textual 'not_applicable', not a missing value.
        _audit_csv(p)
        audited.append(str(p))

    # Every full five-seed checkpoint must exist.
    for seed in FINAL_SEEDS:
        _require_file(CHECKPOINT_DIR / f"EnerDR_Full_seed{seed}.pt")
    for name in BASELINE_NAMES:
        for seed in BASELINE_SEEDS:
            _require_file(CHECKPOINT_DIR / f"Baseline_{name}_seed{seed}.pt")

    # Core ensemble predictions for all three datasets must exist.
    for p in [
        PREDICTION_DIR / "APTOS_test_Full_ensemble.csv",
        PREDICTION_DIR / "Messidor_external_Full_ensemble.csv",
        PREDICTION_DIR / "IDRiD_external_Full_ensemble.csv",
    ]:
        _audit_csv(p)

    catalog_rows = []
    for root, kind in [(TABLE_DIR, "table"), (NEW_TABLE_DIR, "extended_table"),
                       (FIGURE_DIR, "figure"), (NEW_FIGURE_DIR, "extended_figure"),
                       (PREDICTION_DIR, "prediction")]:
        for p in sorted(root.glob("*")):
            if p.is_file():
                catalog_rows.append({"type": kind, "file": str(p), "bytes": int(p.stat().st_size)})
    catalog = pd.DataFrame(catalog_rows)
    catalog.to_csv(OUT_DIR / "complete_result_catalog.csv", index=False)

    audit = {
        "status": "PASS",
        "required_tables_checked": len(required_tables),
        "baseline_models": BASELINE_NAMES,
        "training_seeds": FINAL_SEEDS,
        "all_numeric_outputs_finite": True,
        "all_required_result_files_present": True,
        "result_catalog": str(OUT_DIR / "complete_result_catalog.csv"),
    }
    (OUT_DIR / "complete_run_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2))
    return audit, catalog


COMPLETE_RUN_AUDIT, COMPLETE_RESULT_CATALOG = run_complete_output_audit()
